# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **导入库**: 导入课程库和数据处理所需的库。
2.  **环境配置**: 设置常量和路径。
3.  **辅助函数**: 定义要用到的辅助函数
4.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)，将数据分割为训练集和验证集。
5.  **单个模型**: 用于测试和比较单个模型。
6.  **模型融合**: 用于融合多个模型。包括生成提交文件。
7.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。

## 1. 导入必要的库

In [2]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import numpy as np
import os
import json
from tqdm import tqdm

from sklearn.model_selection import KFold
from sklearn.preprocessing import QuantileTransformer
from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.Recommender_import_list import *
from RecSys_Course_AT_PoliMi.Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask



C:\Users\IceCould\.conda\envs\RecSysFramework\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


Tensorflow is not available


## 2. 项目配置与常量

In [3]:
# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'model_output'
SUBMISSION_FOLDER = 'temp_output' # 提交文件的保存目录

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

print("项目配置加载完成.")

项目配置加载完成.


## 3. 辅助函数

### 3.1 加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.

In [4]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all


### 3.2 接收评估结果字典和模型名称，并以清晰的格式打印关键指标.

In [5]:
def print_results_formatted(results_df, model_name: str = "Model"):
    """
    接收评估结果 DataFrame 和模型名称，并以清晰的格式打印关键指标.

    Args:
        results_df (pd.DataFrame): 来自 Evaluator.evaluateRecommender() 的结果DataFrame.
        model_name (str): 要打印的模型名称.
    """
    # 检查 DataFrame 是否为空，或者指定的 cutoff 值是否作为索引存在
    if results_df.empty or EVALUATION_CUTOFF not in results_df.index:
        print(f"--- 在 cutoff={EVALUATION_CUTOFF} 处没有找到 '{model_name}' 的评估结果 ---")
        return

    # 使用 .loc[] 通过索引名来选取我们关心的那一行数据
    # 这会返回一个 Pandas Series，其行为非常像一个字典
    res_series = results_df.loc[EVALUATION_CUTOFF]

    print(f"--- 模型评估结果: {model_name} ---")
    print(f"--------------------------------------------------")
    # 现在在一个 Pandas Series 上使用 .get() 方法，这是安全且正确的
    print(f"{f'RECALL@{EVALUATION_CUTOFF}':<25}: {res_series.get('RECALL', -1):.4f}   <-- 竞赛官方指标")
    print(f"{f'PRECISION@{EVALUATION_CUTOFF}':<25}: {res_series.get('PRECISION', -1):.4f}")
    print(f"{f'MAP@{EVALUATION_CUTOFF}':<25}: {res_series.get('MAP', -1):.4f}")
    print(f"{f'HIT_RATE@{EVALUATION_CUTOFF}':<25}: {res_series.get('HIT_RATE', -1):.4f}")
    print(f"{f'ITEM_COVERAGE@{EVALUATION_CUTOFF}':<25}: {res_series.get('COVERAGE_ITEM', -1):.4f}")
    print(f"{f'AVG_POPULARITY@{EVALUATION_CUTOFF}':<25}: {res_series.get('AVERAGE_POPULARITY', -1):.4f}")
    print(f"--------------------------------------------------\n")

### 3.3 5-Kold 交叉验证

In [ ]:
def evaluate_with_cross_validation(recommender_class, urm_all, k=5, recommender_params={}):
    """
    使用 K-Fold 交叉验证来评估一个推荐模型。

    Args:
        recommender_class: 要评估的推荐器类 (e.g., ItemKNNCFRecommender).
        urm_all (sps.csr_matrix): 全量的用户-物品交互矩阵.
        k (int): 折数 (e.g., 5).
        recommender_params (dict): 传递给推荐器 fit 方法的超参数.

    Returns:
        float: K 次评估的平均 RECALL@20 分数.
    """
    # KFold 会将数据分割成 k 个部分
    # 我们这里分割的是交互记录的索引
    kfold = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)

    # 存储每一折的 recall 分数
    recall_scores = []

    # 获取非零元素的行和列索引，代表了所有的交互
    rows, cols = urm_all.nonzero()

    print(f"--- 开始 {k}-Fold 交叉验证 ({recommender_class.RECOMMENDER_NAME}) ---")

    # fold_num 是从 1 开始的计数器
    for fold_num, (train_indices, validation_indices) in enumerate(kfold.split(rows), 1):
        print(f"--- 第 {fold_num}/{k} 折 ---")

        # 1. 根据索引创建训练集和验证集
        urm_train = sps.csr_matrix((np.ones(len(train_indices)), (rows[train_indices], cols[train_indices])), shape=urm_all.shape)
        urm_validation = sps.csr_matrix((np.ones(len(validation_indices)), (rows[validation_indices], cols[validation_indices])), shape=urm_all.shape)

        # 2. 初始化评估器
        evaluator = EvaluatorHoldout(urm_validation, cutoff_list=[EVALUATION_CUTOFF])

        # 3. 训练模型
        recommender = recommender_class(urm_train)
        recommender.fit(**recommender_params)

        # 4. 评估模型
        results_df, _ = evaluator.evaluateRecommender(recommender)

        # 5. 提取并存储 recall
        recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)
        recall_scores.append(recall)

        print(f"第 {fold_num}/{k} 折 RECALL@{EVALUATION_CUTOFF}: {recall:.4f}")

    # 6. 计算并返回平均分
    mean_recall = np.mean(recall_scores)
    print(f"\n--- 交叉验证完成 ---")
    print(f"平均 RECALL@{EVALUATION_CUTOFF}: {mean_recall:.4f}\n")

    return mean_recall

### 3.4 读取模型

In [6]:
def load_best_model(recommender_class, urm_train, file_name, modelfile_path):
    """
    加载一个由超参数搜索脚本保存的最佳模型。
    """
    file_path = os.path.join(modelfile_path, file_name)

    print(f"--- 正在加载预训练模型: {file_name} ---")

    # 检查模型文件是否存在
    if not os.path.exists(file_path + ".zip"):
        print(f">>> 警告: 模型文件 '{file_path}.zip' 不存在!")
        print(">>> 请确保超参数优化已完成，并且文件名正确。")
        return None

    # 1. 初始化一个“空”的模型对象
    recommender_instance = recommender_class(urm_train)

    # 2. 调用 .load_model() 方法加载数据
    recommender_instance.load_model(folder_path=modelfile_path, file_name=file_name)

    print("模型加载成功！\n")
    return recommender_instance

## 4. 分割数据集
本节将加载数据，并将其分割为训练集和验证集，为模型评估做准备。

In [7]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)

--- 正在分割数据用于本地验证... ---
数据分割完成.
训练集维度: (27095, 6969), 验证集维度: (27095, 6969)

EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
评估器初始化完成.


## 5. 模型训练与实验
在这里训练、评估和比较不同的推荐模型。

#### 5a. 运行更稳健的评估方法: K-Fold 交叉验证

In [ ]:
# --- 使用交叉验证来评估你的 ItemKNNCF 模型 ---
itemknn_params = {'topK': 100, 'shrink': 10, 'similarity': 'cosine'}
# 运行交叉验证
avg_recall_itemknn = evaluate_with_cross_validation(ItemKNNCFRecommender, urm_all, k=5, recommender_params=itemknn_params)

#### 5b. 运行模型以获得输出

In [ ]:
modelfile_name = "EASE_R_Recommender_best_model"
modelfile_path = "each_model_parameter_3"
recommender_loaded = load_best_model(EASE_R_Recommender, URM_train, modelfile_name, modelfile_path)

if recommender_loaded:
    results_df, _ = evaluator_validation.evaluateRecommender(recommender_loaded)
    print_results_formatted(results_df, "Loaded")


### 5.1 基线模型: Item-based KNN CF

In [ ]:
print("--- 正在训练基线模型 (ItemKNNCF)... ---")
# 初始化模型实例
recommender_itemknn = ItemKNNCFRecommender(URM_train)

# 训练模型 (拟合相似度矩阵)
# 这些是模型的超参数，是后续优化的重点
recommender_itemknn.fit(topK=100, shrink=10, similarity='cosine')
print("模型训练完成.\n")

# 评估模型
results_dict_itemknn, _ = evaluator_validation.evaluateRecommender(recommender_itemknn)

# 打印格式化的结果
print_results_formatted(results_dict_itemknn, "ItemKNNCF Baseline")

### 5.2 实验模型: P3alpha (Code)

In [ ]:
print("--- 正在训练实验模型 (P3alpha)... ---")
# 初始化模型实例
recommender_p3alpha = P3alphaRecommender(URM_train)

# 训练模型
# P3alpha 在隐式反馈数据集上通常表现优异
recommender_p3alpha.fit(topK=100, alpha=0.8, implicit=True)
print("模型训练完成.\n")

# 评估模型
results_dict_p3alpha, _ = evaluator_validation.evaluateRecommender(recommender_p3alpha)

# 打印格式化的结果
print_results_formatted(results_dict_p3alpha, "P3alpha")

### 5.3 实验模型: SLIM BPR (Sparse LInear Methods with BPR)

In [ ]:
print("--- 正在训练实验模型 (SLIM BPR)... ---")
# 初始化模型实例
# 注意：SLIM BPR 是用 Cython 实现的，以获得更好的性能
recommender_slim_bpr = SLIM_BPR_Cython(URM_train)

# 训练模型
# SLIM BPR 是一个机器学习模型，它有 epochs, learning_rate 等参数
# 下面是一组常用的初始参数，是后续超参数优化的重点
recommender_slim_bpr.fit(
    epochs=100,          # 迭代次数
    topK=150,            # 邻域大小
    lambda_i=0.001,      # L2正则化系数 (items)
    lambda_j=0.001,      # L2正则化系数 (users)
    learning_rate=0.01
)
print("模型训练完成.\n")

# 评估模型
# 注意：我们将结果保存在一个新的变量中，以便后续比较和融合
results_df_slim_bpr, _ = evaluator_validation.evaluateRecommender(recommender_slim_bpr)

# 打印格式化的结果
print_results_formatted(results_df_slim_bpr, "SLIM BPR")

### 5.4 实验模型: IALS (Implicit Alternating Least Squares)

In [ ]:
print("--- 正在训练实验模型 (IALS)... ---")
# 初始化模型实例
recommender_ials = IALSRecommender(URM_train)

# 训练模型
# IALS 也是一个迭代式的机器学习模型
# num_factors 是最重要的超参数之一，它决定了潜在空间的维度

IALSRecommender_model_config = {
    "num_factors": 110, "epochs": 20, "confidence_scaling": "linear", "alpha": 11.269227704770643, "epsilon": 2.0344199271132757, "reg": 0.01
}

recommender_ials.fit(**IALSRecommender_model_config)
print("模型训练完成.\n")
model_filename = f"model_recommender_ials_early_stop"
recommender_ials.save_model(folder_path=OUTPUT_FOLDER, file_name=model_filename)
# 评估模型
results_df_ials, _ = evaluator_validation.evaluateRecommender(recommender_ials)

# 打印格式化的结果
print_results_formatted(results_df_ials, "IALS")

### 5.5 实验模型：SLIMElasticNetRecommender (使用局部最优参数)

In [ ]:
print("--- 正在使用最优参数训练冠军模型 (SLIMElasticNet)... ---")

# 初始化模型实例
recommender_slim_elastic = SLIMElasticNetRecommender(URM_train)

# 训练模型
# 使用我们在超参数搜索中找到的最佳参数
# 注意：即使有最优参数，这个模型的训练过程也可能需要较长时间 (几十分钟甚至更久)
recommender_slim_elastic.fit(
    topK = 1000,
    l1_ratio = 0.029739176029882,
    alpha = 0.001
)

print("模型训练完成。\n")
model_filename = f"final_model_recommender_slim_80p"
recommender_slim_elastic.save_model(folder_path=MODEL_FOLDER, file_name=model_filename)

# 评估模型
results_df_slim_elastic, _ = evaluator_validation.evaluateRecommender(recommender_slim_elastic)

# 打印格式化的结果
print_results_formatted(results_df_slim_elastic, "SLIMElasticNet (Optimized)")

## 6. 模型融合

### 6a 双模型融合

#### 6a.1 定义融合推荐器

In [10]:

# MaxMin
class HybridScoreRecommender(BaseRecommender):
    """
    一个通用的混合推荐器，通过加权平均多个模型的分数来进行推荐。
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridScoreRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 初始化一个零分矩阵
        final_scores = np.zeros((len(user_id_array), self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            # 获取单个模型的分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # 归一化分数
            scores = self._normalize_scores(scores)

            # 加权求和
            final_scores += self.weights[i] * scores

        return final_scores

    def _normalize_scores(self, scores):
        """对分数矩阵的每一行进行 Min-Max 归一化"""
        # 检查是否是稀疏矩阵，如果是，则转换为稠密数组
        if sps.issparse(scores):
            scores = scores.toarray()

        max_val = scores.max(axis=1, keepdims=True)
        min_val = scores.min(axis=1, keepdims=True)

        # 避免除以零
        denominator = max_val - min_val
        denominator[denominator == 0] = 1.0

        return (scores - min_val) / denominator

print("模型融合工具已准备就绪。")

模型融合工具已准备就绪。


In [ ]:
# RankGauss
import numpy as np
import scipy.sparse as sps
from scipy.special import erfinv # 必须引入这个逆误差函数

class HybridScoreRecommender(BaseRecommender):
    """
    使用 RankGauss 归一化进行模型融合的混合推荐器
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridScoreRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 初始化最终分数矩阵
        final_scores = None

        for i, recommender in enumerate(self.recommenders):
            # 1. 计算原始分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # 2. 执行 RankGauss 归一化
            # 注意：这一步会改变分数的分布形状，使其服从高斯分布
            scores = self._rank_gauss_score(scores)

            # 3. 加权求和
            if final_scores is None:
                final_scores = self.weights[i] * scores
            else:
                final_scores += self.weights[i] * scores

        return final_scores

    def _rank_gauss_score(self, scores):
        """
        核心方法：对分数矩阵的【每一行】(Per-User) 进行 RankGauss 变换
        """
        # 1. 确保是稠密矩阵 (Rank 操作在稀疏矩阵上很麻烦且容易出错)
        if sps.issparse(scores):
            scores = scores.toarray()

        # 2. 获取排名 (Rank)
        # argsort 两次可以得到每个元素在当前行中的排名 (0 到 n_items-1)
        # 第一次 argsort: 得到排序后的索引
        # 第二次 argsort: 得到每个元素对应的排名
        ranks = np.argsort(np.argsort(scores, axis=1), axis=1)

        # 3. 归一化排名到 (-1, 1) 区间
        n_items = scores.shape[1]

        # 这里的 epsilon 是为了防止出现 erfinv(1) 或 erfinv(-1) 导致的无穷大
        epsilon = 1e-6

        # 将排名映射到 [0, 1]
        r = ranks / (n_items - 1)

        # 线性映射到 [-1, 1]
        r = (r - 0.5) * 2

        # 截断边界，防止无穷大
        r = np.clip(r, -1 + epsilon, 1 - epsilon)

        # 4. 应用逆误差函数 (Inverse Error Function)
        # 这就是 RankGauss 的核心，将均匀分布映射为高斯分布
        gauss_scores = erfinv(r)

        return gauss_scores

In [ ]:
# Rank Normalization
import numpy as np
import scipy.sparse as sps

class HybridScoreRecommender(BaseRecommender):
    """
    使用 Rank Normalization (排名归一化) 的混合推荐器。
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridScoreRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 初始化最终分数矩阵
        final_scores = np.zeros((len(user_id_array), self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            # 1. 获取单个模型的分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # 2. 归一化 (这里调用改好的 Rank Norm)
            scores = self._normalize_scores(scores)

            # 3. 加权求和
            final_scores += self.weights[i] * scores

        return final_scores

    def _normalize_scores(self, scores):
        """
        对分数矩阵的每一行进行 Rank Normalization。
        将分数转换为 0.0 ~ 1.0 的相对排名。
        """
        # 必须转换为稠密矩阵才能进行 argsort
        if sps.issparse(scores):
            scores = scores.toarray()

        # argsort 两次可以得到每个元素的排名索引
        # 第一次: 得到排序后的索引
        # 第二次: 得到原始位置对应的排名值 (0 到 n_items-1)
        # 注意：默认是升序，所以分数越低排名越小，分数越高排名越大
        ranks = np.argsort(np.argsort(scores, axis=1), axis=1)

        # 归一化到 [0, 1]
        n_items = scores.shape[1]

        # 避免分母为0 (虽然 n_items 通常很大)
        if n_items > 1:
            return ranks / (n_items - 1)
        else:
            return np.ones_like(ranks, dtype=np.float32)

print("HybridScoreRecommender (Rank Norm 版) 已定义。")

#### 6a.2 加载已经训练好的两个最佳模型

In [11]:
# --- 1. 加载模型 ---
# 定义模型所在的文件夹
COMBINE_MODEL_FOLDER = "temp_output"

print("--- 正在加载用于融合的基模型... ---")

# 加载 SLIMElasticNetRecommender
recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="SLIMElasticNetRecommender_best_model_best",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 加载 MultVAERecommender
recommender_vea = load_best_model(
    recommender_class=MultVAERecommender_PyTorch_OptimizerMask,
    urm_train=URM_train,
    file_name="multvae_best_trial_14_recall_0.253894-145",
    modelfile_path=OUTPUT_FOLDER
)

# 确保两个模型都成功加载
if recommender_slim is None or recommender_vea is None:
    print(">>> 错误：一个或多个基模型加载失败，无法继续进行融合。")
else:
    print("所有基模型均已成功加载！")

--- 正在加载用于融合的基模型... ---
--- 正在加载预训练模型: SLIMElasticNetRecommender_best_model_best ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model_best'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: multvae_best_trial_14_recall_0.253894-145 ---
模型加载成功！

所有基模型均已成功加载！


#### 6a.3 网格搜索寻找最佳融合权重

In [16]:
# --- 2. 网格搜索寻找最佳融合权重 ---

best_recall = 0
best_alpha = 0
alpha_values = np.arange(0.5, 0.55, 0.05) # 以0.05为步长，进行更精细的搜索

# 确保模型已成功加载
if recommender_slim and recommender_vea:
    print("\n--- 开始在本地验证集上寻找最佳融合权重 alpha ---")

    # 将模型实例放入一个列表中
    recommender_list = [recommender_slim, recommender_vea]

    for alpha in tqdm(alpha_values, desc="搜索 Alpha 权重"):
        # alpha 是第一个模型的权重, (1-alpha) 是第二个模型的权重
        weights = [alpha, 1 - alpha]

        # 创建混合推荐器实例
        hybrid_recommender = HybridScoreRecommender(URM_train, recommender_list, weights)

        # 在验证集上评估
        results_df, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
        current_recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

        if current_recall > best_recall:
            best_recall = current_recall
            best_alpha = alpha
            print(f"发现新的最佳 Recall: {best_recall:.5f} (当 alpha = {best_alpha:.2f})")

    print(f"\n--- 搜索完成！---")
    print(f"最佳 Alpha (SLIMElasticNet 的权重): {best_alpha:.2f}")
    print(f"VEA 的权重: {(1 - best_alpha):.2f}")
    print(f"融合后在本地验证集上的最佳 Recall@20: {best_recall:.5f}")

搜索 Alpha 权重:   0%|          | 0/2 [00:00<?, ?it/s]


--- 开始在本地验证集上寻找最佳融合权重 alpha ---
EvaluatorHoldout: Processed 27058 (100.0%) in 15.87 sec. Users per second: 1705


搜索 Alpha 权重:  50%|█████     | 1/2 [00:15<00:15, 15.90s/it]

发现新的最佳 Recall: 0.29009 (当 alpha = 0.50)
EvaluatorHoldout: Processed 27058 (100.0%) in 15.08 sec. Users per second: 1795


搜索 Alpha 权重: 100%|██████████| 2/2 [00:31<00:00, 15.51s/it]


--- 搜索完成！---
最佳 Alpha (SLIMElasticNet 的权重): 0.50
VEA 的权重: 0.50
融合后在本地验证集上的最佳 Recall@20: 0.29009


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# ==========================================
# 1. 准备工作
# ==========================================
# 召回阶段截断数：设大一点，给精排留空间
CANDIDATE_SIZE = 500

# 获取验证集目标用户
target_users = np.unique(evaluator_validation.URM_test.indices)

# 准备 Ground Truth (用于计算 Recall)
validation_gt = {}
urm_test_coo = evaluator_validation.URM_test.tocoo()
for u, i in zip(urm_test_coo.row, urm_test_coo.col):
    if u not in validation_gt: validation_gt[u] = set()
    validation_gt[u].add(i)

# ==========================================
# 2. 第一阶段：使用 IALS 生成候选集 (Candidate Generation)
# ==========================================
print(f">>> [Stage 1] 使用 IALS 生成每个用户 Top {CANDIDATE_SIZE} 候选集...")

def generate_candidates(recommender, target_users, cutoff):
    user_list = []
    item_list = []
    score_list = []

    batch_size = 1000
    for start_idx in tqdm(range(0, len(target_users), batch_size)):
        end_idx = min(start_idx + batch_size, len(target_users))
        batch_users = target_users[start_idx:end_idx]

        # 1. 预测并取 Top K (自动过滤已读)
        # 这一步决定了我们的“召回上限”，IALS 必须尽可能把用户喜欢的包含在内
        recommendations = recommender.recommend(batch_users, cutoff=cutoff, remove_seen_flag=True)

        # 2. 获取对应的原始分数
        # 这里为了效率，我们只拿推荐出来的物品分数
        all_scores = recommender._compute_item_score(batch_users)

        for i, u_id in enumerate(batch_users):
            rec_items = recommendations[i]
            if len(rec_items) > 0:
                # 获取分数
                rec_scores = all_scores[i, rec_items]

                user_list.extend([u_id] * len(rec_items))
                item_list.extend(rec_items)
                score_list.extend(rec_scores)

    return pd.DataFrame({
        "user_id": user_list,
        "item_id": item_list,
        "ials_score": score_list
    })

# 生成候选 DataFrame
df_cascade = generate_candidates(recommender_b, target_users, CANDIDATE_SIZE)
print(f"候选集生成完毕。总行数: {len(df_cascade)}")

# ==========================================
# 3. 第二阶段：使用 SLIM 对候选集打分 (Re-scoring)
# ==========================================
print(f">>> [Stage 2] 使用 SLIM 对候选集进行二次打分...")

# 为了快速给 df_cascade 里的 (user, item) 打分，
# 我们不需要让 SLIM 预测 Top K，只需要让它“填空”

# 预先计算 SLIM 的分数矩阵通常太大了，我们按 batch 处理 DataFrame
# 但为了代码最简化，我们这里使用一种 trick：
# 遍历用户，单独拿出该用户在 df_cascade 里的 items，让 SLIM 打分

# 优化：为了向量化，我们还是按 batch 走，但是利用 mask
slim_scores_list = []
batch_size = 1000
unique_users = df_cascade['user_id'].unique()

# 建立一个快速查找表: user -> [item_indices in dataframe]
# 这步稍微有点繁琐，为了简单，我们采用另一种高效方案：
# 直接计算 batch 用户的 SLIM 分数，然后用 merge 匹配

# 分批计算 SLIM 分数并 Merge
temp_dfs = []
for start_idx in tqdm(range(0, len(unique_users), batch_size)):
    end_idx = min(start_idx + batch_size, len(unique_users))
    batch_users = unique_users[start_idx:end_idx]

    # 计算 SLIM 全量分数 (注意：这里 SLIM 是稀疏的，很多是 0)
    batch_scores = recommender_slim._compute_item_score(batch_users)
    if sps.issparse(batch_scores):
        batch_scores = batch_scores.toarray()

    # 我们只关心 df_cascade 里存在的那些 user-item 对
    # 为了避免复杂的索引操作，我们把 batch_scores 转成 long format 再 merge
    # 但全量转 long 太慢了。

    # === 高效路径 ===
    # 提取当前 batch 在 df_cascade 中的子集
    mask = df_cascade['user_id'].isin(batch_users)
    subset_df = df_cascade[mask].copy()

    # 构造一个映射: user_id -> batch_index
    u_map = {u: i for i, u in enumerate(batch_users)}

    # 提取分数
    # subset_df['user_id'] -> batch row index
    # subset_df['item_id'] -> col index

    row_indices = subset_df['user_id'].map(u_map).values
    col_indices = subset_df['item_id'].values

    # 直接从矩阵取值
    subset_df['slim_score'] = batch_scores[row_indices, col_indices]
    temp_dfs.append(subset_df)

# 重组 DataFrame
df_cascade = pd.concat(temp_dfs, ignore_index=True)

# 填充 SLIM 可能产生的 NaN (虽理论上不该有)
df_cascade['slim_score'] = df_cascade['slim_score'].fillna(0.0)

print("SLIM 打分完成。")
print(df_cascade.head())

# ==========================================
# 4. 组内归一化 (Group-wise Normalization)
# ==========================================
print(">>> [Stage 3] 对 Top-K 候选集进行组内归一化...")

# 注意：现在的 min 和 max 是基于【IALS 选出来的这 200 个物品】
# 这符合“精排”的逻辑：我们在有限的候选集里比高低
def normalize_group(df, col):
    g = df.groupby('user_id')[col]
    g_min = g.transform('min')
    g_max = g.transform('max')
    diff = g_max - g_min
    diff = diff.replace(0, 1.0)
    return (df[col] - g_min) / diff

df_cascade['ials_norm'] = normalize_group(df_cascade, 'ials_score')
df_cascade['slim_norm'] = normalize_group(df_cascade, 'slim_score')

# ==========================================
# 5. 搜索最佳权重
# ==========================================
print(">>> [Stage 4] 搜索最佳混合权重...")

def calculate_recall_cascade(df, top_k=20):
    # 排序：按混合分
    df_sorted = df.sort_values(['user_id', 'final_score'], ascending=[True, False])
    # 截断
    df_top = df_sorted.groupby('user_id').head(top_k)

    # 聚合
    preds = df_top.groupby('user_id')['item_id'].apply(list).to_dict()

    hits = 0
    total = 0

    # 遍历所有目标用户 (严格模式)
    for u in target_users:
        if u not in validation_gt: continue
        gt_items = validation_gt[u]
        if len(gt_items) == 0: continue

        total += 1
        recs = preds.get(u, [])
        hits += len(set(recs) & gt_items) / len(gt_items)

    return hits / total if total > 0 else 0

best_recall = 0
best_alpha = 0
# IALS 是召回源，通常保留较高权重，或者 SLIM 作为强力辅助
alpha_values = np.arange(0.5, 0.9, 0.05)

for alpha in tqdm(alpha_values):
    # 混合公式: alpha * SLIM + (1-alpha) * IALS
    df_cascade['final_score'] = alpha * df_cascade['slim_norm'] + (1-alpha) * df_cascade['ials_norm']

    recall = calculate_recall_cascade(df_cascade, top_k=20)

    if recall > best_recall:
        best_recall = recall
        best_alpha = alpha
        print(f"New Best Recall: {best_recall:.5f} (alpha={alpha:.2f})")

print(f"\n最佳结果: Recall={best_recall:.5f}, SLIM_Weight={best_alpha}")

# 看看如果不重排，纯 IALS 的效果是多少 (alpha=0)
# 看看如果不考虑 IALS 分数，纯用 SLIM 对 IALS 结果重排是多少 (alpha=1)

In [ ]:
import numpy as np
import scipy.sparse as sps

class TieredHybridRecommender(BaseRecommender):
    """
    分层策略推荐器：
    - Top N (比如前10个): 严格由 SLIM 决定 (High Precision)。
    - Rest (比如后10个): 由 融合模型 决定 (High Recall)。
    """
    def __init__(self, urm_train, recommender_slim, recommender_ials, slim_weight, top_n_slim=10):
        super(TieredHybridRecommender, self).__init__(urm_train)
        self.recommender_slim = recommender_slim
        self.recommender_ials = recommender_ials
        self.alpha = slim_weight
        self.top_n_slim = top_n_slim # 前多少个位置完全留给 SLIM

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 1. 获取两个模型的原始分数
        scores_slim = self.recommender_slim._compute_item_score(user_id_array, items_to_compute)
        scores_ials = self.recommender_ials._compute_item_score(user_id_array, items_to_compute)

        if sps.issparse(scores_slim): scores_slim = scores_slim.toarray()
        if sps.issparse(scores_ials): scores_ials = scores_ials.toarray()

        # 2. 归一化 (User-wise Min-Max) - 必须步骤
        # SLIM 归一化
        min_s = scores_slim.min(axis=1, keepdims=True)
        max_s = scores_slim.max(axis=1, keepdims=True)
        den_s = max_s - min_s
        den_s[den_s == 0] = 1.0
        norm_slim = (scores_slim - min_s) / den_s

        # IALS 归一化
        min_i = scores_ials.min(axis=1, keepdims=True)
        max_i = scores_ials.max(axis=1, keepdims=True)
        den_i = max_i - min_i
        den_i[den_i == 0] = 1.0
        norm_ials = (scores_ials - min_i) / den_i

        # 3. 计算基础融合分数 (Hybrid Score)
        # 这是所有物品的“底色”，范围 [0, 1]
        hybrid_scores = self.alpha * norm_slim + (1 - self.alpha) * norm_ials

        # 4. 【核心逻辑】提取 SLIM 的 Top N 并“置顶”
        # 我们使用 argpartition 快速找到每行最大的 Top N 个索引
        # 注意：argpartition 不保证内部顺序，所以我们只是圈定这 N 个物品
        k = min(self.top_n_slim, self.n_items - 1)

        # 找到 SLIM 分数最高的 Top K 个物品的索引
        # axis=1 表示按用户操作
        top_k_indices = np.argpartition(norm_slim, -k, axis=1)[:, -k:]

        # 5. 构造最终分数
        # 复制一份 hybrid_scores 作为最终结果
        final_scores = hybrid_scores.copy()

        # 6. 施加魔法：Boosting
        # 我们希望这 K 个物品排在最前面，且内部顺序由 SLIM 决定。
        # 做法：让它们的分数 = norm_slim (0~1) + 10.0 (Huge Constant)
        # 这样它们的分数变成了 10.x ~ 11.x，远远大于其他物品的 0.x ~ 1.0

        # 使用 numpy 的 fancy indexing
        rows = np.arange(len(user_id_array))[:, None]

        # 提取这些位置原本的 SLIM 分数
        original_slim_scores_topk = norm_slim[rows, top_k_indices]

        # 覆盖写入：Top K 位置的分数 = SLIM分数 + 10
        # 这样排序后，前 K 名就是 SLIM 的 Top K，且顺序也是 SLIM 的顺序
        # 第 K+1 名开始，就是 Hybrid 分数最高的那些
        final_scores[rows, top_k_indices] = original_slim_scores_topk + 10.0

        return final_scores

In [ ]:
from tqdm import tqdm

# 设定 SLIM 霸榜的前几名
TOP_N_SLIM = 15

print(f"\n--- 开始搜索 (策略: Top {TOP_N_SLIM} SLIM, Rest Hybrid) ---")

best_recall = 0
best_alpha = 0
alpha_values = np.arange(0.6, 0.9, 0.05) # 步长 0.1

for alpha in tqdm(alpha_values, desc="Search"):

    # 实例化我们的分层推荐器
    tiered_recommender = TieredHybridRecommender(
        URM_train,
        recommender_slim,
        recommender_b,
        slim_weight=alpha,
        top_n_slim=TOP_N_SLIM
    )

    # 评估
    results_df, _ = evaluator_validation.evaluateRecommender(tiered_recommender)
    current_recall = results_df.loc[20]["RECALL"]

    print(f"Alpha: {alpha:.2f} | Recall@20: {current_recall:.5f}")

    if current_recall > best_recall:
        best_recall = current_recall
        best_alpha = alpha
        print(f"New Best! -> {best_recall:.5f}")

print(f"最佳配置: Top {TOP_N_SLIM} SLIM, 后续混合权重 alpha={best_alpha}")

In [ ]:
import numpy as np
import scipy.sparse as sps

class HybridRankRecommender(BaseRecommender):
    """
    基于倒数排名 (Reciprocal Rank) 的混合推荐器
    Score = alpha * (1 / (rank_1 + 1)) + (1 - alpha) * (1 / (rank_2 + 1))
    这种方法对异常分数不敏感，只关注相对顺序。
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridRankRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        n_users = len(user_id_array)
        # 最终分数矩阵
        final_scores = np.zeros((n_users, self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            # 1. 计算原始分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)
            if sps.issparse(scores): scores = scores.toarray()

            # 2. 将分数转换为排名 (Rank)
            # argsort 得到从小到大的索引，再 argsort 得到排名
            # 注意：我们需要的是“分数越大，排名越小(0, 1, 2...)”
            # 这里的 argsort().argsort() 得到的是升序排名 (最小分是0, 最大分是N)
            # 所以我们要对 -scores 进行排序，或者反转结果

            # 方法：对负分数排序 -> 0就是最高分，1是第二高分
            ranks = np.argsort(np.argsort(-scores, axis=1), axis=1)

            # 3. 计算倒数排名分数 (Reciprocal Rank)
            # Rank 0 -> Score 1.0
            # Rank 1 -> Score 0.5
            # ...
            rr_scores = 1.0 / (ranks + 1.0)

            # 4. 加权融合
            final_scores += self.weights[i] * rr_scores

        return final_scores

# --- 搜索代码 ---
from tqdm import tqdm

print(f"\n--- 开始搜索 (基于 Rank Fusion) ---")

best_recall = 0
best_alpha = 0
# Rank Fusion 通常对权重不那么敏感，但也值得一搜
alpha_values = np.arange(0.4, 0.6, 0.05)

recommender_list = [recommender_slim, recommender_vea]

for alpha in tqdm(alpha_values, desc="Searching Alpha"):
    weights = [alpha, 1 - alpha]

    # 使用新的 Rank Recommender
    hybrid_recommender = HybridRankRecommender(URM_train, recommender_list, weights)

    results_df, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    current_recall = results_df.loc[20]["RECALL"]

    print(f"Alpha: {alpha:.2f} | Recall@20: {current_recall:.5f}")

    if current_recall > best_recall:
        best_recall = current_recall
        best_alpha = alpha
        print(f"New Best! -> {best_recall:.5f}")

print(f"最佳结果: {best_recall} (alpha={best_alpha})")

#### 6a.4 使用最佳权重生成提交文件

In [ ]:
# --- 1. 配置最佳权重和模型信息 ---
BEST_ALPHA = 0.43 # !!! 请确保将此值替换为您在网格搜索中找到的最佳 alpha !!!

MODEL_FOLDER = 'model_output' # 模型文件所在的文件夹

# --- 2. 加载在 *全量数据* 上训练好的模型 ---
print("\n--- 正在加载在全量数据上预训练的最终模型... ---")

# 加载 SLIMElasticNetRecommender
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 加载 b
final_recommender_b = load_best_model(
    recommender_class=MultVAERecommender_PyTorch_OptimizerMask,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="29-MultVAE_Best_Model_Sub",
    modelfile_path=MODEL_FOLDER
)


# --- 3. 创建最终的混合推荐器并生成提交 ---
if final_recommender_slim and final_recommender_b:

    final_recommender_list = [final_recommender_slim, final_recommender_b]
    final_weights = [BEST_ALPHA, 1 - BEST_ALPHA]

    final_hybrid_recommender = HybridScoreRecommender(urm_all, final_recommender_list, final_weights)

    # --- 开始生成推荐 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终融合推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    submission_filename = f"submission_FINAL_Hybrid_SLIM_IALS_alpha_{BEST_ALPHA:.2f}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename) # SUBMISSION_FOLDER 在之前已定义

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
else:
    print("\n>>> 错误：一个或多个最终模型加载失败，无法生成提交文件。")

### 6b 高斯融合

#### 6b.1 加载模型

In [ ]:
# --- 1. 加载模型 ---

print("--- 正在加载用于三模型融合的基模型... ---")

# 加载 SLIMElasticNetRecommender
recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train,
    file_name="model_recommender_slim_80p",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 IALSRecommender
recommender_b = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train,
    file_name="IALSRecommender_best_model_best",
    modelfile_path=OUTPUT_FOLDER
)

# 确保所有模型都成功加载
if not all([recommender_slim, recommender_b,]):
    print(">>> 错误：一个或多个基模型加载失败，无法继续进行三模型融合。")
else:
    print("所有基模型均已成功加载！准备进行权重搜索。")

#### 6b.2 高斯融合的准备

In [ ]:
# --- 2. 准备阶段：生成候选集并提取原始分数 ---

# 设定每个模型提取的候选数量 (越大越好，建议 100-200)
CANDIDATE_CUTOFF = 200

def get_candiates_dataframe(recommender, target_users):
    """
    使用模型对 target_users 进行预测，并返回格式化的 DataFrame。
    带有详细的进度打印。
    """
    user_list = []
    item_list = []
    score_list = []

    # 这里设置 batch_size 是为了防止内存爆掉，同时也方便显示进度条
    batch_size = 1000

    # 分批次处理
    for start_idx in tqdm(range(0, len(target_users), batch_size), desc=f"正在提取 {recommender.RECOMMENDER_NAME} 的分数"):
        end_idx = min(start_idx + batch_size, len(target_users))
        batch_users = target_users[start_idx:end_idx]

        # 1. 推荐 (返回 items)
        # remove_seen_flag=True 会自动利用模型内部存储的 URM_train 过滤已读
        recommendations = recommender.recommend(
            batch_users,
            cutoff=CANDIDATE_CUTOFF,
            remove_seen_flag=True
        )

        # 2. 获取分数
        # 注意：这里假设 recommender 有 _compute_item_score 方法
        # 如果没有，可能需要检查你的框架 recommend 是否直接返回 (items, scores)
        all_scores_batch = recommender._compute_item_score(batch_users)

        for i, user_id in enumerate(batch_users):
            rec_items = recommendations[i]
            # 从密集矩阵中只提取推荐物品的分数
            rec_scores = all_scores_batch[i, rec_items]

            user_list.extend([user_id] * len(rec_items))
            item_list.extend(rec_items)
            score_list.extend(rec_scores)

    df = pd.DataFrame({
        "user_id": user_list,
        "item_id": item_list,
        "score": score_list
    })

    return df

# --- 验证流程开始 ---

print(">>> 步骤 1: 准备目标用户列表")
# 获取验证集用户 (和之前逻辑一样)
target_users_list = np.unique(evaluator_validation.URM_test.indices)
print(f"验证集目标用户数量: {len(target_users_list)}")


print("\n>>> 步骤 2: 提取 SLIM 模型分数")
# 临时给模型加个名字属性，方便打印
recommender_slim.RECOMMENDER_NAME = "SLIM"
df_slim = get_candiates_dataframe(recommender_slim, target_users_list)
df_slim.rename(columns={"score": "slim_score"}, inplace=True)

# --- [检查点 1] ---
print("\n--- [检查] SLIM 输出预览 ---")
print(f"数据形状: {df_slim.shape}")
print(df_slim.head(5))
print(f"SLIM 分数范围: Min={df_slim['slim_score'].min():.4f}, Max={df_slim['slim_score'].max():.4f}")
print("----------------------------")


print("\n>>> 步骤 3: 提取 IALS 模型分数")
recommender_b.RECOMMENDER_NAME = "IALS"
df_ials = get_candiates_dataframe(recommender_b, target_users_list)
df_ials.rename(columns={"score": "ials_score"}, inplace=True)

# --- [检查点 2] ---
print("\n--- [检查] IALS 输出预览 ---")
print(f"数据形状: {df_ials.shape}")
print(df_ials.head(5))
print(f"IALS 分数范围: Min={df_ials['ials_score'].min():.4f}, Max={df_ials['ials_score'].max():.4f}")
print("----------------------------")


print("\n>>> 步骤 4: 合并两个模型的结果 (Outer Join)")
# Outer Join 保证只要有一个模型推荐了，我们就保留下来
df_hybrid = pd.merge(df_slim, df_ials, on=["user_id", "item_id"], how="outer")

# --- [检查点 3] ---
print("\n--- [检查] 合并后的数据 (填充前) ---")
print(f"合并后总行数: {len(df_hybrid)}")
print("前5行 (可能包含 NaN):")
print(df_hybrid.head(5))
print("\n缺失值统计 (验证模型互补性):")
print(df_hybrid.isnull().sum())
print("----------------------------")


print("\n>>> 步骤 5: 填充缺失值 (Fill NaNs)")
# 策略：用该模型在当前数据中的最小值 - 1.0 来填充
# 这样保证没被推荐的物品，在排序时一定排在最后
slim_min = df_hybrid["slim_score"].min()
ials_min = df_hybrid["ials_score"].min()

print(f"SLIM 填充值: {slim_min} - 1.0 = {slim_min - 1.0}")
print(f"IALS 填充值: {ials_min} - 1.0 = {ials_min - 1.0}")

df_hybrid["slim_score"].fillna(slim_min - 1.0, inplace=True)
df_hybrid["ials_score"].fillna(ials_min - 1.0, inplace=True)

# --- [检查点 4] ---
print("\n--- [检查] 填充后的最终数据 ---")
print(df_hybrid.head(5))
print(f"现在是否还有 NaN? : {df_hybrid.isnull().any().any()}")
print("----------------------------")

print("\n>>> 准备阶段完成！数据已准备好进行高斯化和网格搜索。")

#### 6b.3 高斯变换

In [ ]:
# --- 3. 实施 RankGauss 变换 ---

print("--- 执行 RankGauss 高斯化变换 ---")

def apply_rank_gauss(df, col_name):
    """
    对指定列进行 RankGauss 高斯化变换。
    """
    # 修正1: n_quantiles 设为 10000。
    # 这意味着我们将数据分布切分成 10000 个精细的刻度，这对于推荐排序来说精度已经绰绰有余。
    # 不再使用 len(df)，避免了你遇到的 "quantiles > samples" 错误。

    # 修正2: subsample 设为 1e9 (10亿)。
    # 默认是 1e5 (10万)，我们把它改大，强制 sklearn 使用你这 190 万行的所有数据来计算分布，
    # 而不是只随机采样一部分。这样结果更稳定。

    # 修正3: 使用全局随机种子 RANDOM_SEED
    transformer = QuantileTransformer(
        n_quantiles=10000,
        output_distribution='normal',
        random_state=RANDOM_SEED,
        subsample=1e9
    )

    raw_scores = df[col_name].values.reshape(-1, 1)

    # 拟合并变换
    transformed_scores = transformer.fit_transform(raw_scores)

    # 存回 DataFrame
    df[col_name + "_gauss"] = transformed_scores.flatten()
    return df

# 对两列分数分别进行高斯化
df_hybrid = apply_rank_gauss(df_hybrid, "slim_score")
df_hybrid = apply_rank_gauss(df_hybrid, "ials_score")

print("高斯化完成！分数已映射到标准正态分布。")

#### 6b.4 网格搜索

In [ ]:
# 为了计算 Recall，我们需要验证集的 Ground Truth
# 构建一个简单的字典: user_id -> set(true_items)
validation_gt = {}
urm_test_coo = evaluator_validation.URM_test.tocoo()
for u, i in zip(urm_test_coo.row, urm_test_coo.col):
    if u not in validation_gt:
        validation_gt[u] = set()
    validation_gt[u].add(i)

def calculate_recall_at_20_fast(df_scored):
    """
    快速计算 Recall@20
    """
    # 按融合分降序排列
    # 技巧：我们只对每个用户的前20个感兴趣，使用 groupby + head 可以加速，
    # 但最快的方法是直接利用 pandas 的排序

    # 1. 排序
    df_sorted = df_scored.sort_values(["user_id", "final_score"], ascending=[True, False])

    # 2. 取 Top 20
    df_top20 = df_sorted.groupby("user_id").head(20)

    # 3. 计算命中数
    hits = 0
    total_relevant = 0

    # 这里的循环可以用 merge 优化，但为了代码易读性先用循环
    # 实际上对于几万用户，这个循环在几秒内能跑完
    current_user = -1
    recs = []

    # 将 DataFrame 转为 numpy 数组加速
    user_arr = df_top20["user_id"].values
    item_arr = df_top20["item_id"].values

    # 统计 Recall
    # 这是一个简化版的评估，主要为了速度
    # 如果 evaluator_validation 很快，你也可以要把结果转回去用它测

    hits_count = 0
    total_rel_count = 0

    # 这一步稍微有点慢，但比重新 recommend 快无数倍
    unique_users = np.unique(user_arr)

    # 优化：通过 GroupBy 聚合
    grouped = df_top20.groupby('user_id')['item_id'].apply(list)

    cumulative_recall = 0.0
    valid_users_count = 0

    for u_id, recommended_items in grouped.items():
        if u_id in validation_gt:
            true_items = validation_gt[u_id]
            rel_count = len(true_items)
            if rel_count > 0:
                # 计算交集
                hit = len(set(recommended_items) & true_items)
                cumulative_recall += hit / rel_count
                valid_users_count += 1

    return cumulative_recall / valid_users_count if valid_users_count > 0 else 0


print("\n--- 开始搜索最佳权重 (基于 RankGauss) ---")

best_recall = 0
best_alpha = 0
alpha_values = np.arange(0.5, 0.9, 0.05) # 因为 RankGauss 后不需要太精细，0.05 足够

for alpha in tqdm(alpha_values, desc="Search Alpha"):
    # 线性加权：注意使用的是 _gauss 后缀的列
    # Score = w * SLIM_Gauss + (1-w) * IALS_Gauss
    df_hybrid["final_score"] = (alpha * df_hybrid["slim_score_gauss"] +
                                (1 - alpha) * df_hybrid["ials_score_gauss"])

    # 计算 Recall
    current_recall = calculate_recall_at_20_fast(df_hybrid)

    if current_recall > best_recall:
        best_recall = current_recall
        best_alpha = alpha
        print(f"New Best Recall: {best_recall:.5f} (alpha={best_alpha:.2f})")

print(f"\n--- 最终结果 ---")
print(f"最佳 RankGauss 权重 alpha: {best_alpha}")
print(f"对应的 Recall@20: {best_recall}")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# ==========================================
# 1. 准备验证集 Ground Truth (保持不变)
# ==========================================
# 假设 evaluator_validation 已经在上下文中
validation_gt = {}
urm_test_coo = evaluator_validation.URM_test.tocoo()
for u, i in zip(urm_test_coo.row, urm_test_coo.col):
    if u not in validation_gt:
        validation_gt[u] = set()
    validation_gt[u].add(i)

# ==========================================
# 2. 核心修改：按用户分组归一化函数
# ==========================================
def normalize_scores_per_user(df, score_col):
    """
    模拟代码 B 的逻辑：
    对 DataFrame 中的每个 user_id，单独计算该用户分数的 min 和 max，
    然后进行 min-max 归一化。
    """
    # 使用 transform 可以直接将聚合结果广播回原 DataFrame 的形状，速度很快
    # 计算每个用户的 Max 和 Min
    user_max = df.groupby("user_id")[score_col].transform("max")
    user_min = df.groupby("user_id")[score_col].transform("min")

    # 计算分母
    denominator = user_max - user_min

    # 防止分母为 0 (即该用户只有一个推荐结果，或所有结果分数相同)
    # 如果分母为0，我们设为1，这样分子是0，结果就是0，符合逻辑
    denominator = denominator.replace(0, 1.0)

    # 计算归一化后的分数
    return (df[score_col] - user_min) / denominator

# ==========================================
# 3. 数据预处理
# ==========================================
print("正在对分数进行 [用户级] Min-Max 归一化 (匹配 Recommender 类逻辑)...")

# 假设你的列名是 'slim_score' 和 'ials_score'
# 生成新列：slim_score_norm, ials_score_norm
df_hybrid["slim_score_norm"] = normalize_scores_per_user(df_hybrid, "slim_score")
df_hybrid["ials_score_norm"] = normalize_scores_per_user(df_hybrid, "ials_score")

# 填充潜在的 NaN (虽然理论上有了 replace(0, 1) 不会有 NaN，但为了保险)
df_hybrid["slim_score_norm"] = df_hybrid["slim_score_norm"].fillna(0)
df_hybrid["ials_score_norm"] = df_hybrid["ials_score_norm"].fillna(0)

print("归一化完成。")

# ==========================================
# 4. 快速评估函数 (保持不变)
# ==========================================
def calculate_recall_at_20_fast(df_scored):
    # 1. 排序
    df_sorted = df_scored.sort_values(["user_id", "final_score"], ascending=[True, False])
    # 2. 取 Top 20
    df_top20 = df_sorted.groupby("user_id").head(20)
    # 3. 聚合
    grouped = df_top20.groupby('user_id')['item_id'].apply(list)

    cumulative_recall = 0.0
    valid_users_count = 0

    for u_id, recommended_items in grouped.items():
        if u_id in validation_gt:
            true_items = validation_gt[u_id]
            rel_count = len(true_items)
            if rel_count > 0:
                hit = len(set(recommended_items) & true_items)
                cumulative_recall += hit / rel_count
                valid_users_count += 1

    return cumulative_recall / valid_users_count if valid_users_count > 0 else 0

# ==========================================
# 5. 网格搜索最佳 Alpha
# ==========================================
print("\n--- 开始搜索最佳权重 (基于 Per-User Min-Max) ---")

best_recall = 0
best_alpha = 0
# 因为是用户级归一化，结果比较稳定，可以搜细一点
alpha_values = np.arange(0.5, 0.9, 0.05)

for alpha in tqdm(alpha_values, desc="Search Alpha"):
    # 使用 _norm 后缀的列进行加权
    df_hybrid["final_score"] = (alpha * df_hybrid["slim_score_norm"] +
                                (1 - alpha) * df_hybrid["ials_score_norm"])

    current_recall = calculate_recall_at_20_fast(df_hybrid)

    if current_recall > best_recall:
        best_recall = current_recall
        best_alpha = alpha
        print(f"New Best Recall: {best_recall:.5f} (alpha={best_alpha:.2f})")

print(f"\n--- 最终结果 ---")
print(f"最佳权重 alpha: {best_alpha}")
print(f"对应的 Recall@20: {best_recall}")

In [ ]:
def calculate_recall_at_20_fast(df_scored):
    """
    快速计算 Recall@20
    """
    # 按融合分降序排列
    # 技巧：我们只对每个用户的前20个感兴趣，使用 groupby + head 可以加速，
    # 但最快的方法是直接利用 pandas 的排序

    # 1. 排序
    df_sorted = df_scored.sort_values(["user_id", "final_score"], ascending=[True, False])

    # 2. 取 Top 20
    df_top20 = df_sorted.groupby("user_id").head(20)

    # 3. 计算命中数
    hits = 0
    total_relevant = 0

    # 这里的循环可以用 merge 优化，但为了代码易读性先用循环
    # 实际上对于几万用户，这个循环在几秒内能跑完
    current_user = -1
    recs = []

    # 将 DataFrame 转为 numpy 数组加速
    user_arr = df_top20["user_id"].values
    item_arr = df_top20["item_id"].values

    # 统计 Recall
    # 这是一个简化版的评估，主要为了速度
    # 如果 evaluator_validation 很快，你也可以要把结果转回去用它测

    hits_count = 0
    total_rel_count = 0

    # 这一步稍微有点慢，但比重新 recommend 快无数倍
    unique_users = np.unique(user_arr)

    # 优化：通过 GroupBy 聚合
    grouped = df_top20.groupby('user_id')['item_id'].apply(list)

    cumulative_recall = 0.0
    valid_users_count = 0

    for u_id, recommended_items in grouped.items():
        if u_id in validation_gt:
            true_items = validation_gt[u_id]
            rel_count = len(true_items)
            if rel_count > 0:
                # 计算交集
                hit = len(set(recommended_items) & true_items)
                cumulative_recall += hit / rel_count
                valid_users_count += 1

    return cumulative_recall / valid_users_count if valid_users_count > 0 else 0

In [ ]:
print("\n--- 尝试方案 B: MinMax 归一化融合 ---")
from sklearn.preprocessing import MinMaxScaler

# 1. 实施 MinMax (映射到 0~1)
scaler = MinMaxScaler()
df_hybrid["slim_mm"] = scaler.fit_transform(df_hybrid["slim_score"].values.reshape(-1, 1))
df_hybrid["ials_mm"] = scaler.fit_transform(df_hybrid["ials_score"].values.reshape(-1, 1))

# 2. 网格搜索
best_recall = 0
best_w = 0

for w in tqdm(np.arange(0.0, 1.01, 0.05), desc="搜索 MinMax 权重"):
    # 普通加权
    df_hybrid["final_score"] = w * df_hybrid["slim_mm"] + (1 - w) * df_hybrid["ials_mm"]

    current_recall = calculate_recall_at_20_fast(df_hybrid)

    if current_recall > best_recall:
        best_recall = current_recall
        best_w = w
        print(f"发现新高 (MinMax): {best_recall:.5f} (w={best_w:.2f})")

In [ ]:
print("\n--- 尝试方案 C: MinMax + 幂次增强 (非线性) ---")
from sklearn.preprocessing import MinMaxScaler

# 1. 基础 MinMax (0~1)
scaler = MinMaxScaler()
df_hybrid["slim_mm"] = scaler.fit_transform(df_hybrid["slim_score"].values.reshape(-1, 1))
df_hybrid["ials_mm"] = scaler.fit_transform(df_hybrid["ials_score"].values.reshape(-1, 1))

# 2. 关键步骤：对强模型施加“幂次”
# 尝试让 SLIM 变得更“尖锐” (平方)，让 IALS 保持原样或变得更“平缓”
# 这里的 2.0 是经验值，意味着我们超级信任 SLIM 的高分段
df_hybrid["slim_mm_poly"] = np.power(df_hybrid["slim_mm"], 2.0)
df_hybrid["ials_mm_poly"] = df_hybrid["ials_mm"] # IALS 保持线性，或者试一下 np.power(..., 0.8)

best_recall = 0
best_w = 0

# 搜索
for w in tqdm(np.arange(0.5, 1.01, 0.05), desc="搜索 Power 权重"):
    # 融合
    df_hybrid["final_score"] = w * df_hybrid["slim_mm_poly"] + (1 - w) * df_hybrid["ials_mm_poly"]

    current_recall = calculate_recall_at_20_fast(df_hybrid)

    if current_recall > best_recall:
        best_recall = current_recall
        best_w = w
        print(f"发现新高 (Power): {best_recall:.5f} (w={best_w:.2f})")

In [ ]:
print("\n--- 尝试方案 D: Z-Score 标准化 ---")
from sklearn.preprocessing import StandardScaler

# 1. StandardScaler (均值0，方差1)
scaler = StandardScaler()
df_hybrid["slim_std"] = scaler.fit_transform(df_hybrid["slim_score"].values.reshape(-1, 1))
df_hybrid["ials_std"] = scaler.fit_transform(df_hybrid["ials_score"].values.reshape(-1, 1))

best_recall = 0
best_w = 0

for w in tqdm(np.arange(0.0, 1.01, 0.05), desc="搜索 Z-Score 权重"):
    # 融合
    df_hybrid["final_score"] = w * df_hybrid["slim_std"] + (1 - w) * df_hybrid["ials_std"]

    current_recall = calculate_recall_at_20_fast(df_hybrid)

    if current_recall > best_recall:
        best_recall = current_recall
        best_w = w
        print(f"发现新高 (Z-Score): {best_recall:.5f} (w={best_w:.2f})")

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

print("--- 准备 Top-K 锁定实验 ---")

# 1. 重现目前最好的融合分 (MinMax, w=0.85)
# 如果你之前运行过 MinMax 代码，df_hybrid 里应该已经有 slim_mm 和 ials_mm 了
# 如果没有，请取消下面两行的注释重新生成
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler()
# df_hybrid["slim_mm"] = scaler.fit_transform(df_hybrid["slim_score"].values.reshape(-1, 1))
# df_hybrid["ials_mm"] = scaler.fit_transform(df_hybrid["ials_score"].values.reshape(-1, 1))

best_w_minmax = 0.85
df_hybrid["base_hybrid_score"] = (best_w_minmax * df_hybrid["slim_mm"] +
                                  (1 - best_w_minmax) * df_hybrid["ials_mm"])

# 2. 计算 SLIM 的“组内排名” (User-wise Rank)
# 这一步很关键，我们要知道对于每个用户，哪个电影是 SLIM 的第1名，哪个是第2名...
print("正在计算 SLIM 的用户组内排名...")
# 先按 SLIM 分数降序排
df_hybrid = df_hybrid.sort_values(["user_id", "slim_score"], ascending=[True, False])
# 使用 cumcount 生成排名 (0, 1, 2...) -> 变成 (1, 2, 3...)
df_hybrid["slim_rank_user"] = df_hybrid.groupby("user_id").cumcount() + 1

# 3. 搜索最佳 K 值
print("\n--- 开始搜索最佳保送名额 K (Top-K Locking) ---")

best_recall = 0
best_k = 0

# 我们尝试保送 SLIM 的前 1 名 到 前 10 名
for k in range(10, 16):
    # 复制一份基础分
    # 这里的逻辑是：如果该物品是 SLIM 的前 K 名，给它加一个巨大的分 (比如 100.0)
    # 这样排序时它一定在最前面。非 Top-K 的物品保持原有的 Hybrid 分数。

    # 使用 np.where 向量化操作，速度极快
    # 逻辑：如果 rank <= k, 分数 = 100.0 + 原分 (其实原分都不重要了); 否则 = base_hybrid_score
    current_final_scores = np.where(
        df_hybrid["slim_rank_user"] <= k,
        100.0 + df_hybrid["base_hybrid_score"],
        df_hybrid["base_hybrid_score"]
    )

    # 临时把这个分数放进 DataFrame 测 Recall
    df_hybrid["score_k_lock"] = current_final_scores

    # 复用之前的快速计算函数 (请确保 calculate_recall_at_20_fast 函数还在内存里)
    # 注意：这里我们要按 score_k_lock 排序
    # 为了速度，我们需要修改 calculate_recall_at_20_fast 让它支持指定列名，或者我们手动改名
    # 这里简单起见，我把列名改成 final_score 传进去
    df_temp = df_hybrid[["user_id", "item_id", "score_k_lock"]].copy()
    df_temp.rename(columns={"score_k_lock": "final_score"}, inplace=True)

    current_recall = calculate_recall_at_20_fast(df_temp)

    print(f"Top-{k} 锁定 SLIM -> Recall: {current_recall:.5f}")

    if current_recall > best_recall:
        best_recall = current_recall
        best_k = k

print(f"\n--- 最终结果 ---")
print(f"最佳策略: 强制保送 SLIM 的前 {best_k} 名")
print(f"剩余位置 (20-{best_k}) 由 MinMax(0.85) 融合填充")
print(f"最终 Recall@20: {best_recall:.5f}")

### 临时测试

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

# 加载 SLIMElasticNetRecommender
recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 UserKNNCFRecommender
recommender_user = load_best_model(
    recommender_class=UserKNNCFRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="UserKNNCFRecommender_cosine_best_model",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 IALSRecommender
recommender_ials = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train,
    file_name="model_recommender_ials_80p",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 EASE_R_Recommender
recommender_ease = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=URM_train,
    file_name="EASE_R_Recommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

target_users_list = np.unique(evaluator_validation.URM_test.indices)

df_slim = get_candiates_dataframe(recommender_slim, target_users_list)
df_slim.rename(columns={"score": "slim_score"}, inplace=True)

df_user = get_candiates_dataframe(recommender_user, target_users_list)
df_user.rename(columns={"score": "user_score"}, inplace=True)

df_ials = get_candiates_dataframe(recommender_ials, target_users_list)
df_ials.rename(columns={"score": "ials_score"}, inplace=True)

df_ease = get_candiates_dataframe(recommender_ease, target_users_list)
df_ease.rename(columns={"score": "ease_score"}, inplace=True)

print("合并四模型数据...")
df_hybrid = pd.merge(df_slim, df_ease, on=["user_id", "item_id"], how="outer")
df_hybrid = pd.merge(df_hybrid, df_ials, on=["user_id", "item_id"], how="outer")
df_hybrid = pd.merge(df_hybrid, df_user, on=["user_id", "item_id"], how="outer")

# 备份一份原始数据，方便后面反复使用
df_raw = df_hybrid.copy()
print(f"数据准备完成，共 {len(df_raw)} 行候选数据。")

In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("--- Stacking v2.0: 数据准备 ---")

# 1. 确保你有包含三个模型原始分数的 DataFrame (df_hybrid)
# 之前的代码里你应该已经有了这个 df_hybrid，包含 slim_score, ials_score, user_score
# 且已经填充了缺失值 (fillna 0 或 min)

# 备份一下，以免改坏
train_df = df_hybrid.copy()

# 2. 构造 Label (Ground Truth)
# 我们需要知道候选集里哪些是用户真的看过的
# validation_gt 是一个字典: user_id -> set(item_ids) (之前的代码里有)
# 如果没有，请重新运行:
validation_gt = {}
urm_test_coo = evaluator_validation.URM_test.tocoo()
for u, i in zip(urm_test_coo.row, urm_test_coo.col):
    if u not in validation_gt: validation_gt[u] = set()
    validation_gt[u].add(i)

def get_label(row):
    if row['user_id'] in validation_gt:
        return 1 if row['item_id'] in validation_gt[row['user_id']] else 0
    return 0

# 这步可能稍微慢一点，但很必要
print("正在打标签 (Labeling)...")
# 优化速度：向量化操作
# 将 gt 转换为 set of tuples 加速查找
gt_pairs = set(zip(urm_test_coo.row, urm_test_coo.col))
train_df['label'] = train_df.apply(lambda x: 1 if (x['user_id'], x['item_id']) in gt_pairs else 0, axis=1)

print(f"正样本数量: {train_df['label'].sum()}")

In [ ]:
print("--- Stacking v2.0: 特征工程 ---")

# 1. 添加排名特征 (Rank)
# 树模型对数值大小不敏感，但对相对顺序很敏感
train_df['slim_rank'] = train_df.groupby('user_id')['slim_score'].rank(ascending=False, method='min')
train_df['ials_rank'] = train_df.groupby('user_id')['ials_score'].rank(ascending=False, method='min')
train_df['user_rank'] = train_df.groupby('user_id')['user_score'].rank(ascending=False, method='min')

# 2. 添加归一化特征 (MinMax)
# 之前的 MinMax 已经证明有效，加入进去
scaler = MinMaxScaler()
train_df['slim_mm'] = scaler.fit_transform(train_df['slim_score'].values.reshape(-1, 1))
train_df['ials_mm'] = scaler.fit_transform(train_df['ials_score'].values.reshape(-1, 1))
train_df['user_mm'] = scaler.fit_transform(train_df['user_score'].values.reshape(-1, 1))

# 3. 添加差值特征 (Interactions)
# 比如：SLIM 和 IALS 的分歧有多大？
train_df['diff_slim_user'] = train_df['slim_mm'] - train_df['user_mm']

# 定义特征列表
feature_cols = [
    'slim_score', 'ials_score', 'user_score',
    'slim_rank', 'ials_rank', 'user_rank',
    'slim_mm', 'ials_mm', 'user_mm',
    'diff_slim_user'
]

print("特征准备完毕。")

In [ ]:
print("--- Stacking v2.0: 训练 LGBMRanker ---")

# 1. 排序
# LGBM 要求同一个 Group 的数据必须物理上在一起
train_df = train_df.sort_values(by=['user_id'], ascending=True)

# 2. 准备 Group 信息
# qid (query id) 对应的样本数量
group_counts = train_df.groupby('user_id').size().to_frame('count')['count'].values

# 3. 切分训练集和验证集 (用于 Early Stopping)
# 这里我们再次切分这 20% 的数据：80% 用来训练 Stacker，20% 用来验证 Stacker 防止过拟合
# 注意：不能按行切，要按用户切！

unique_users = train_df['user_id'].unique()
np.random.shuffle(unique_users)
train_users_num = int(len(unique_users) * 0.8)

stack_train_users = unique_users[:train_users_num]
stack_valid_users = unique_users[train_users_num:]

# 划分 DataFrame
df_stack_train = train_df[train_df['user_id'].isin(stack_train_users)]
df_stack_valid = train_df[train_df['user_id'].isin(stack_valid_users)]

# 准备 Group
group_train = df_stack_train.groupby('user_id').size().values
group_valid = df_stack_valid.groupby('user_id').size().values

# 准备 X, y
X_train = df_stack_train[feature_cols]
y_train = df_stack_train['label']

X_valid = df_stack_valid[feature_cols]
y_valid = df_stack_valid['label']

# 4. 定义模型
# lambdarank 是专门做排序的
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",         # 或者 map
    n_estimators=500,      # 树的数量
    learning_rate=0.05,    # 学习率
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED
)

# 5. 训练
print("开始训练 Ranker...")
ranker.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_valid, y_valid)],
    eval_group=[group_valid],
    eval_at=[20],          # 重点关注 Top 20 的效果
)

print("训练完成！")

In [ ]:
import matplotlib.pyplot as plt

# 查看特征重要性
print("\n特征重要性:")
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranker.feature_importances_
}).sort_values('importance', ascending=False)
print(importance)

# 预测并评估最终 Recall
# 我们用训练好的 Ranker 对所有验证集用户重新打分
all_pred = ranker.predict(train_df[feature_cols])
train_df['final_stack_score'] = all_pred

print("\n计算 Stacking 后的 Recall@20...")
# 1. 创建一个用于评估的临时副本，以免搞乱原始数据
df_eval = train_df.copy()

# 2. 强制覆盖：直接把 stacking 的预测分赋值给 final_score
# 这样无论之前有没有 final_score，现在都只有这一列，且是 stacking 的结果
df_eval['final_score'] = df_eval['final_stack_score']

# 3. 计算 Recall
recall = calculate_recall_at_20_fast(df_eval)
print(f"Stacking Recall: {recall:.5f}")

#### 提交过程

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'temp_output'
SUBMISSION_PATH = 'temp_output/SUBMISSION.csv' # 提交文件的保存目录

print("--- 最终提交生成阶段 ---")

# 1. 确定目标用户 (主办方要求的测试集用户)
# 通常在 submission_sample.csv 或 data_reader 里能找到
# 假设 URM_test_full 是包含所有用户的矩阵，或者你有一个 target_user_list
# 这里假设 target_users_list 包含了所有需要预测的用户ID
# target_users_list = ...

print(f"目标用户数: {len(target_users_list)}")

# 加载 SLIMElasticNetRecommender
recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all,
    file_name="model_recommender_slim_80p",
    modelfile_path=MODEL_FOLDER
)

# 加载 UserKNNCFRecommender
recommender_user = load_best_model(
    recommender_class=UserKNNCFRecommender,
    urm_train=urm_all,
    file_name="UserKNNCFRecommender_cosine_best_model",
    modelfile_path=MODEL_FOLDER
)

# 加载 IALSRecommender
recommender_ials = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all,
    file_name="model_recommender_ials_80p",
    modelfile_path=MODEL_FOLDER
)

# --- 配置 ---
CANDIDATE_CUTOFF = 200  # 候选集大小，建议 >= 100

# 检查模型
if recommender_slim is None:
    raise ValueError(">>> 错误：recommender_slim加载失败，请检查路径。")

if recommender_ials is None:
    raise ValueError(">>> 错误：recommender_ials加载失败，请检查路径。")

if recommender_user is None:
    raise ValueError(">>> 错误：recommender_user加载失败，请检查路径。")

# --- STEP 2: 准备目标用户和候选集生成函数 ---

# 加载目标用户
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values
print(f"目标用户数量: {len(target_user_ids)}")

# 复用之前的候选集提取函数 (稍作修改以适应提交阶段)
def get_candiates_dataframe(recommender, target_users):
    user_list = []
    item_list = []
    score_list = []

    # 批处理以防内存溢出，同时显示进度
    batch_size = 2000

    for start_idx in tqdm(range(0, len(target_users), batch_size), desc=f"提取 {recommender.RECOMMENDER_NAME}"):
        end_idx = min(start_idx + batch_size, len(target_users))
        batch_users = target_users[start_idx:end_idx]

        # 推荐 Top N (自动过滤已读)
        recommendations = recommender.recommend(
            batch_users,
            cutoff=CANDIDATE_CUTOFF,
            remove_seen_flag=True
        )

        # 获取分数
        all_scores_batch = recommender._compute_item_score(batch_users)

        for i, user_id in enumerate(batch_users):
            rec_items = recommendations[i]
            rec_scores = all_scores_batch[i, rec_items]

            user_list.extend([user_id] * len(rec_items))
            item_list.extend(rec_items)
            score_list.extend(rec_scores)

    return pd.DataFrame({
        "user_id": user_list,
        "item_id": item_list,
        "score": score_list
    })

# --- STEP 3: 生成并合并候选集 ---
print("\n--- 3. 生成候选集 (Test Candidates) ---")

# 1. 提取 SLIM
df_slim = get_candiates_dataframe(recommender_slim, target_user_ids)
df_slim.rename(columns={"score": "slim_score"}, inplace=True)

# 2. 提取 IALS
df_ials = get_candiates_dataframe(recommender_ials, target_user_ids)
df_ials.rename(columns={"score": "ials_score"}, inplace=True)

# 3. 提取 UserKNN
df_user = get_candiates_dataframe(recommender_user, target_user_ids)
df_user.rename(columns={"score": "user_score"}, inplace=True)

print("正在合并三个模型的候选集...")
# Outer Join 保证不漏掉任何一个模型的高分推荐
test_df = pd.merge(df_slim, df_ials, on=["user_id", "item_id"], how="outer")
test_df = pd.merge(test_df, df_user, on=["user_id", "item_id"], how="outer")

# 填充缺失值 (填 0.0 或一个极小值)
test_df.fillna(0.0, inplace=True)

print(f"候选集生成完毕，共 {len(test_df)} 行。")

# --- STEP 4: 特征工程 (必须与训练时完全一致) ---
print("\n--- 4. 特征工程 ---")

# 1. Rank 特征
print("计算 Rank...")
test_df['slim_rank'] = test_df.groupby('user_id')['slim_score'].rank(ascending=False, method='min')
test_df['ials_rank'] = test_df.groupby('user_id')['ials_score'].rank(ascending=False, method='min')
test_df['user_rank'] = test_df.groupby('user_id')['user_score'].rank(ascending=False, method='min')

# 2. MinMax 特征
print("计算 MinMax...")
scaler = MinMaxScaler()
test_df['slim_mm'] = scaler.fit_transform(test_df['slim_score'].values.reshape(-1, 1))
test_df['ials_mm'] = scaler.fit_transform(test_df['ials_score'].values.reshape(-1, 1))
test_df['user_mm'] = scaler.fit_transform(test_df['user_score'].values.reshape(-1, 1))

# 3. 差值特征
test_df['diff_slim_user'] = test_df['slim_mm'] - test_df['user_mm']

# 定义特征列 (顺序必须与训练 ranker 时一致)
feature_cols = [
    'slim_score', 'ials_score', 'user_score',
    'slim_rank', 'ials_rank', 'user_rank',
    'slim_mm', 'ials_mm', 'user_mm',
    'diff_slim_user'
]

# --- STEP 5: 使用 Stacker 预测并生成文件 ---
print("\n--- 5. Stacker 预测与文件生成 ---")

# 这里的 ranker 是你之前在 notebook 内存里训练好的那个模型对象
# 如果你重启过 notebook，你需要重新运行一遍 Stacking 训练代码 (用 80% 数据训练出 ranker)
if 'ranker' not in locals():
    raise ValueError(">>> 错误：内存中找不到 'ranker' 模型。请确保你没有重启内核，或者请重新运行 LGBMRanker 的训练代码块。")

# 预测
test_df['final_score'] = ranker.predict(test_df[feature_cols])

# 排序并取 Top 20
print("排序并截断 Top 20...")
submission_df = test_df.sort_values(by=['user_id', 'final_score'], ascending=[True, False])
submission_df = submission_df.groupby('user_id').head(20)

# 格式化输出
print("正在写入 CSV 文件...")
grouped_preds = submission_df.groupby('user_id')['item_id'].apply(list)

with open(SUBMISSION_PATH, "w") as f:
    f.write("user_id,item_list\n")
    for user_id in tqdm(target_user_ids):
        if user_id in grouped_preds:
            items = grouped_preds[user_id]
            item_str = " ".join(map(str, items))
        else:
            # 防御性代码：如果某个用户没有任何推荐 (极罕见)，给空列表
            item_str = ""
        f.write(f"{user_id},{item_str}\n")

print(f"\n--- 提交文件已生成: {SUBMISSION_PATH} ---")
print("Good Luck! 🚀")

### 6d 相似度融合

In [7]:
import numpy as np
from Recommenders.BaseRecommender import BaseRecommender

class GeneralizedLinearHybrid(BaseRecommender):
    """
    专门用于融合 EASE 和 SLIM 这种量级差异巨大的模型
    """
    def __init__(self, URM_train, recommenders: list, verbose=True):
        super(GeneralizedLinearHybrid, self).__init__(URM_train, verbose=verbose)
        self.recommenders = recommenders

    def fit(self, alphas: list = None):
        self.alphas = alphas

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 汇总所有模型的预测分数
        item_scores_total = None

        for index, recommender in enumerate(self.recommenders):
            # 获取原始分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # === 关键步骤：动态归一化 (Min-Max per user) ===
            # EASE 的分数范围极大且有负数，SLIM 是 0-1 左右
            # 必须把它们拉到同一个起跑线

            # 1. 每一行的最大值和最小值
            min_scores = scores.min(axis=1, keepdims=True)
            max_scores = scores.max(axis=1, keepdims=True)

            # 2. 归一化到 [0, 1]
            # 加上 1e-6 防止除以零
            norm_scores = (scores - min_scores) / (max_scores - min_scores + 1e-6)

            # 3. 加权累加
            if item_scores_total is None:
                item_scores_total = norm_scores * self.alphas[index]
            else:
                item_scores_total += norm_scores * self.alphas[index]

        return item_scores_total

# ==========================================
# 实战调用
# ==========================================

recommender_slim_val = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train,
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 MultVAERecommender
recommender_vea = load_best_model(
    recommender_class=MultVAERecommender_PyTorch_OptimizerMask,
    urm_train=URM_train,
    file_name="MultVAE_Best_Model",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 IALSRecommender
recommender_ials_val = load_best_model(
    recommender_class=RP3betaRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="RP3betaRecommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

--- 正在加载预训练模型: SLIMElasticNetRecommender_best_model ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: MultVAE_Best_Model ---
模型加载成功！

--- 正在加载预训练模型: RP3betaRecommender_best_model ---
RP3betaRecommender: Loading model from file 'temp_outputRP3betaRecommender_best_model'
RP3betaRecommender: Loading complete
模型加载成功！



In [ ]:
hybrid_model = GeneralizedLinearHybrid(
    URM_train,
    [recommender_slim_val, recommender_vea, recommender_ials_val]
)

print("Starting Grid Search...")
print("-" * 60)
print(f"{'SLIM':<10} {'EASE':<10} {'IALS':<10} | {'RECALL':<10}")
print("-" * 60)

best_recall = 0.0
best_alphas = None
results = []

# 生成权重范围：0.0 到 1.0，步长 0.05
step = 0.05
w_range = np.linspace(0, 1, int(1/step) + 1)

# 遍历 SLIM 的权重
for w_slim in w_range:
    # 遍历 EASE 的权重
    for w_ease in w_range:

        # 剩下给 IALS
        w_ials = 1.0 - w_slim - w_ease

        # === 剪枝策略 (Pruning) ===
        # 1. 权重必须非负
        # 2. 浮点数误差修正 (比如 0.00000001 当作 0)
        if w_ials < -0.001: continue

        # 3. 策略限制：
        #    - SLIM 和 EASE 必须是主力 (至少 0.2)
        #    - IALS 只是辅助，不要超过 0.4
        if w_slim < 0.3 or w_ease < 0.3: continue
        if w_ials > 0.15: continue

        # 此时是一个有效的组合
        current_alphas = [w_slim, w_ease, w_ials]

        # 注入权重并评估
        hybrid_model.fit(alphas=current_alphas)

        # 评估 (这里假设只看 Recall@20)
        result_df, _ = evaluator_validation.evaluateRecommender(hybrid_model)
        current_recall = result_df.loc[20]['RECALL']

        # 记录结果
        results.append((current_alphas, current_recall))

        # 打印当前较好的结果 (减少刷屏，只打印超过 0.28 的)
        if current_recall > 0.28:
            print(f"{w_slim:.2f}       {w_ease:.2f}       {w_ials:.2f}       | {current_recall:.5f}")

        # 更新最佳
        if current_recall > best_recall:
            best_recall = current_recall
            best_alphas = current_alphas

# ==========================================
# 4. 输出最终结果
# ==========================================
print("-" * 60)
print(f"🎉 Best Recall Found: {best_recall:.6f}")
print(f"🏆 Best Alphas: SLIM={best_alphas[0]:.2f}, EASE={best_alphas[1]:.2f}, IALS={best_alphas[2]:.2f}")

# 打印 Top 5 组合，方便观察稳定性
print("\nTop 5 Combinations:")
results.sort(key=lambda x: x[1], reverse=True)
for i in range(min(5, len(results))):
    alpha, score = results[i]
    print(f"#{i+1}: {alpha} -> {score:.6f}")

print(f"Best Alphas: {best_alphas}, Best Recall: {best_recall}")

#### 6d.1 加载模型

In [8]:
recommender_slim_val = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train,
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

recommender_easer_val = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=URM_train,
    file_name="EASE_R_Recommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

# 加载 IALSRecommender
recommender_ials_val = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="IALSRecommender_best_model_best",
    modelfile_path=OUTPUT_FOLDER
)

recommender_userknn_val = load_best_model(
    recommender_class=UserKNNCFRecommender,
    urm_train=URM_train,
    file_name="UserKNNCFRecommender_cosine_best_model",
    modelfile_path=OUTPUT_FOLDER
)

recommender_vea_val = load_best_model(
    recommender_class=MultVAERecommender_PyTorch_OptimizerMask,
    urm_train=URM_train,
    file_name="MultVAE_Best_Model",
    modelfile_path=OUTPUT_FOLDER
)

recommender_rp3_val = load_best_model(
    recommender_class=RP3betaRecommender,
    urm_train=URM_train,
    file_name="RP3betaRecommender_best_model",
    modelfile_path=OUTPUT_FOLDER
)

--- 正在加载预训练模型: SLIMElasticNetRecommender_best_model ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: EASE_R_Recommender_best_model ---
EASE_R_Recommender: Loading model from file 'temp_outputEASE_R_Recommender_best_model'
EASE_R_Recommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: IALSRecommender_best_model_best ---
IALSRecommender: Loading model from file 'temp_outputIALSRecommender_best_model_best'
IALSRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: UserKNNCFRecommender_cosine_best_model ---
UserKNNCFRecommender: Loading model from file 'temp_outputUserKNNCFRecommender_cosine_best_model'
UserKNNCFRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: MultVAE_Best_Model ---
模型加载成功！

--- 正在加载预训练模型: RP3betaRecommender_best_model ---
RP3betaRecommender: Loading model from file 'temp_outputRP3betaRecommender_best_model'
RP3betaRecommender: Loading complete
模型加载

#### 6d.2 相似度矩阵融合

In [112]:
import numpy as np
import pandas as pd

# ==========================================
# 1. 配置模型列表 (确保顺序与权重一一对应)
# ==========================================
# 请确保列表中的模型已经 load_model 完毕
recommender_list = [
    recommender_slim_val,   # [0]
    recommender_easer_val,  # [1]
    recommender_ials_val,   # [2]
    recommender_vea_val,     # [3]
    recommender_rp3_val     # [4]
]

class GeneralizedLinearHybrid(BaseRecommender):
    """
    包含 Min-Max Normalization 的线性加权混合模型
    专门用于解决 EASE 与其他模型量级不一致的问题
    """
    def __init__(self, URM_train, recommenders: list, verbose=True):
        super(GeneralizedLinearHybrid, self).__init__(URM_train, verbose=verbose)
        self.recommenders = recommenders

    def fit(self, alphas: list = None):
        self.alphas = alphas

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        item_scores_total = None

        for index, recommender in enumerate(self.recommenders):
            # 获取原始分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # === Min-Max Normalization ===
            min_scores = scores.min(axis=1, keepdims=True)
            max_scores = scores.max(axis=1, keepdims=True)
            # 加上 1e-6 防止除以零
            norm_scores = (scores - min_scores) / (max_scores - min_scores + 1e-6)

            # 加权累加
            if item_scores_total is None:
                item_scores_total = norm_scores * self.alphas[index]
            else:
                item_scores_total += norm_scores * self.alphas[index]

        return item_scores_total


# 初始化混合模型
# 注意：GeneralizedLinearHybrid 内部通常会对各模型分数先进行归一化再加权
hybrid_model = GeneralizedLinearHybrid(URM_train, recommender_list)

# ==========================================
# 2. 手动输入权重 (Alphas)
# ==========================================
# 权重之和建议为 1.0。
# 你可以根据对 0.01 分差距的直觉，微调这些小数。
manual_alphas = [
    0.42,   # SLIM 权重
    0.00,   # EASE权重
    0.05,   # IALS权重
    0.43,    # VAE权重
    0.00    # RP3权重
]

# 验证权重数量是否匹配
if len(manual_alphas) != len(recommender_list):
    raise ValueError(f"权重数量({len(manual_alphas)})与模型数量({len(recommender_list)})不符！")

# ==========================================
# 3. 运行评估
# ==========================================
print(f"正在测试权重: {manual_alphas}")

# 注入手动权重
hybrid_model.fit(alphas=manual_alphas)

# 执行验证集评估 (Recall@20)
# cutoff 设置为 20 对应比赛要求
result_df, _ = evaluator_validation.evaluateRecommender(hybrid_model)
final_recall = result_df.loc[20]['RECALL']

print("-" * 30)
print(f"📊 手动测试结果:")
for i, weight in enumerate(manual_alphas):
    print(f"模型 [{recommender_list[i].RECOMMENDER_NAME}]: {weight:.4f}")

print(f"\n🏆 本地 Recall@20: {final_recall:.6f}")
print("-" * 30)

正在测试权重: [0.42, 0.0, 0.05, 0.43, 0.0]
EvaluatorHoldout: Processed 27058 (100.0%) in 23.62 sec. Users per second: 1146
------------------------------
📊 手动测试结果:
模型 [SLIMElasticNetRecommender]: 0.4200
模型 [EASE_R_Recommender]: 0.0000
模型 [IALSRecommender]: 0.0500
模型 [MultVAERecommender_PyTorch]: 0.4300
模型 [RP3betaRecommender]: 0.0000

🏆 本地 Recall@20: 0.294412
------------------------------


In [18]:
import numpy as np
import pandas as pd

# ==========================================
# 1. 配置模型列表 (确保顺序与权重一一对应)
# ==========================================
# 请确保列表中的模型已经 load_model 完毕
recommender_list = [
    recommender_slim_val,   # [0]
    recommender_easer_val,  # [1]
    recommender_ials_val,   # [2]
    recommender_vea_val,     # [3]
    recommender_rp3_val     # [4]
]

class GeneralizedLinearHybrid(BaseRecommender):
    """
    包含 Min-Max Normalization 的线性加权混合模型
    专门用于解决 EASE 与其他模型量级不一致的问题
    """
    def __init__(self, URM_train, recommenders: list, verbose=True):
        super(GeneralizedLinearHybrid, self).__init__(URM_train, verbose=verbose)
        self.recommenders = recommenders

    def fit(self, alphas: list = None):
        self.alphas = alphas

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        item_scores_total = None

        for index, recommender in enumerate(self.recommenders):
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # === Min-Max Normalization ===
            min_scores = scores.min(axis=1, keepdims=True)
            max_scores = scores.max(axis=1, keepdims=True)
            norm_scores = (scores - min_scores) / (max_scores - min_scores + 1e-6)

            # 加权累加
            if item_scores_total is None:
                item_scores_total = norm_scores * self.alphas[index]
            else:
                item_scores_total += norm_scores * self.alphas[index]

        return item_scores_total



# 初始化混合模型
# 注意：GeneralizedLinearHybrid 内部通常会对各模型分数先进行归一化再加权
hybrid_model = GeneralizedLinearHybrid(URM_train, recommender_list)

# ==========================================
# 2. 手动输入权重 (Alphas)
# ==========================================
# 你可以根据对 0.01 分差距的直觉，微调这些小数。
manual_alphas = [
    0.42,   # SLIM 权重
    0.00,   # EASE权重
    0.05,   # IALS权重
    0.53,    # VAE权重
    0.00    # RP3权重
]

# 验证权重数量是否匹配
if len(manual_alphas) != len(recommender_list):
    raise ValueError(f"权重数量({len(manual_alphas)})与模型数量({len(recommender_list)})不符！")

# ==========================================
# 3. 运行评估
# ==========================================
print(f"正在测试权重: {manual_alphas}")

# 注入手动权重
hybrid_model.fit(alphas=manual_alphas)

# 执行验证集评估 (Recall@20)
# cutoff 设置为 20 对应比赛要求
result_df, _ = evaluator_validation.evaluateRecommender(hybrid_model)
final_recall = result_df.loc[20]['RECALL']

print("-" * 30)
print(f"📊 手动测试结果:")
for i, weight in enumerate(manual_alphas):
    print(f"模型 [{recommender_list[i].RECOMMENDER_NAME}]: {weight:.4f}")

print(f"\n🏆 本地 Recall@20: {final_recall:.6f}")
print("-" * 30)

正在测试权重: [0.42, 0.0, 0.05, 0.53, 0.0]
EvaluatorHoldout: Processed 27058 (100.0%) in 23.26 sec. Users per second: 1163
------------------------------
📊 手动测试结果:
模型 [SLIMElasticNetRecommender]: 0.4200
模型 [EASE_R_Recommender]: 0.0000
模型 [IALSRecommender]: 0.0500
模型 [MultVAERecommender_PyTorch]: 0.5300
模型 [RP3betaRecommender]: 0.0000

🏆 本地 Recall@20: 0.293274
------------------------------


In [21]:
import numpy as np
import pandas as pd
import itertools


# ==========================================
# 1. 配置模型列表 (确保顺序与权重一一对应)
# ==========================================
# 请确保列表中的模型已经 load_model 完毕
recommender_list = [
    recommender_slim_val,   # [0]
    recommender_easer_val,  # [1]
    recommender_ials_val,   # [2]
    recommender_vea_val,     # [3]
    recommender_rp3_val     # [4]
]

class GeneralizedLinearHybrid(BaseRecommender):
    """
    包含 Min-Max Normalization 的线性加权混合模型
    专门用于解决 EASE 与其他模型量级不一致的问题
    """
    def __init__(self, URM_train, recommenders: list, verbose=True):
        super(GeneralizedLinearHybrid, self).__init__(URM_train, verbose=verbose)
        self.recommenders = recommenders

    def fit(self, alphas: list = None):
        self.alphas = alphas

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        item_scores_total = None

        for index, recommender in enumerate(self.recommenders):
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # === Min-Max Normalization ===
            min_scores = scores.min(axis=1, keepdims=True)
            max_scores = scores.max(axis=1, keepdims=True)
            norm_scores = (scores - min_scores) / (max_scores - min_scores + 1e-6)

            # 加权累加
            if item_scores_total is None:
                item_scores_total = norm_scores * self.alphas[index]
            else:
                item_scores_total += norm_scores * self.alphas[index]

        return item_scores_total



# 初始化混合模型
# 注意：GeneralizedLinearHybrid 内部通常会对各模型分数先进行归一化再加权
hybrid_model = GeneralizedLinearHybrid(URM_train, recommender_list)

# ==========================================
# 2. 参数搜索空间定义
# ==========================================

alpha_space = {
    "SLIM":  {"start": 0.40, "end": 0.50, "step": 0.02},
    "EASE":  {"start": 0.00, "end": 0.00, "step": 1.00},  # 固定为 0
    "IALS":  {"start": 0.03, "end": 0.07, "step": 0.01},
    "VAE":   {"start": 0.40, "end": 0.60, "step": 0.02},
    "RP3":   {"start": 0.00, "end": 0.00, "step": 1.00}   # 固定为 0
}

# 是否要求权重和为 1（建议 False）
ENFORCE_SUM_TO_ONE = False
SUM_TOLERANCE = 1e-6

alpha_candidates = []

for key in ["SLIM", "EASE", "IALS", "VAE", "RP3"]:
    cfg = alpha_space[key]
    values = np.arange(cfg["start"], cfg["end"] + 1e-9, cfg["step"])
    alpha_candidates.append(values)


print("开始参数搜索...")

results = []

for alphas in itertools.product(*alpha_candidates):

    alphas = list(alphas)

    # 可选：约束权重和
    if ENFORCE_SUM_TO_ONE:
        if abs(sum(alphas) - 1.0) > SUM_TOLERANCE:
            continue

    # 注入权重
    hybrid_model.fit(alphas=alphas)

    # 本地评估
    result_df, _ = evaluator_validation.evaluateRecommender(hybrid_model)
    recall = result_df.loc[20]["RECALL"]

    results.append({
        "SLIM": alphas[0],
        "EASE": alphas[1],
        "IALS": alphas[2],
        "VAE":  alphas[3],
        "RP3":  alphas[4],
        "Recall@20": recall
    })

    print(f"alphas={alphas} -> Recall@20={recall:.6f}")


results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Recall@20", ascending=False)

print("\nTop 10 参数组合：")
print(results_df.head(10))

开始参数搜索...
EvaluatorHoldout: Processed 27058 (100.0%) in 22.18 sec. Users per second: 1220
alphas=[0.4, 0.0, 0.03, 0.4, 0.0] -> Recall@20=0.294034
EvaluatorHoldout: Processed 27058 (100.0%) in 22.43 sec. Users per second: 1206
alphas=[0.4, 0.0, 0.03, 0.42000000000000004, 0.0] -> Recall@20=0.294044
EvaluatorHoldout: Processed 27058 (100.0%) in 22.01 sec. Users per second: 1229
alphas=[0.4, 0.0, 0.03, 0.44000000000000006, 0.0] -> Recall@20=0.294163
EvaluatorHoldout: Processed 27058 (100.0%) in 21.79 sec. Users per second: 1242
alphas=[0.4, 0.0, 0.03, 0.4600000000000001, 0.0] -> Recall@20=0.294237
EvaluatorHoldout: Processed 27058 (100.0%) in 22.23 sec. Users per second: 1217
alphas=[0.4, 0.0, 0.03, 0.4800000000000001, 0.0] -> Recall@20=0.294218
EvaluatorHoldout: Processed 27058 (100.0%) in 23.18 sec. Users per second: 1167
alphas=[0.4, 0.0, 0.03, 0.5000000000000001, 0.0] -> Recall@20=0.294261
EvaluatorHoldout: Processed 27058 (100.0%) in 22.57 sec. Users per second: 1199
alphas=[0.4, 0.0,

KeyboardInterrupt: 

### 冷热

In [19]:
class GroupedLinearHybrid(BaseRecommender):
    def __init__(self, URM_train, recommenders, is_hot_mask, item_popularity, verbose=True):
        super(GroupedLinearHybrid, self).__init__(URM_train, verbose=verbose)
        self.recommenders = recommenders
        self.is_hot_mask = is_hot_mask
        # 流行度预处理，这个 log 必须加，否则热门项权重太大直接把分数除没了
        self.item_pop_log = np.log(item_popularity + 2)

    def fit(self, alphas_hot, alphas_cold, lambda_hot=0.0, lambda_cold=0.0):
        self.alphas_hot = np.array(alphas_hot)
        self.alphas_cold = np.array(alphas_cold)
        self.lambda_hot = lambda_hot
        self.lambda_cold = lambda_cold

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        item_scores_total = np.zeros((len(user_id_array), self.n_items))
        batch_is_hot = self.is_hot_mask[user_id_array]

        for index, recommender in enumerate(self.recommenders):
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # --- 你那成功的 Min-Max ---
            min_scores = scores.min(axis=1, keepdims=True)
            max_scores = scores.max(axis=1, keepdims=True)
            norm_scores = (scores - min_scores) / (max_scores - min_scores + 1e-6)

            if index == 3:  # MultVAE index
                norm_scores = norm_scores

            # --- 分群加权 ---
            curr_alphas = np.where(batch_is_hot, self.alphas_hot[index], self.alphas_cold[index])
            item_scores_total += norm_scores * curr_alphas[:, np.newaxis]

        return item_scores_total

In [20]:
import numpy as np
import scipy.sparse as sp
from Evaluation.Evaluator import EvaluatorHoldout

# ==========================================
# 1. 准备数据和划分用户
# ==========================================
URM_validation_all = evaluator_validation.URM_test
user_profile_length = np.ediff1d(URM_train.indptr)
threshold = 50

cold_mask = user_profile_length <= threshold
hot_mask = user_profile_length > threshold

recommender_list = [
    recommender_slim_val,   # [0]
    recommender_easer_val,  # [1]
    recommender_ials_val,   # [2]
    recommender_vea_val,     # [3]
    recommender_rp3_val     # [4]
]

# 顺序: [SLIM, EASE, IALS, VAE, RP3]
alphas_hot = [0.45, 0.00, 0.05, 0.50, 0.00]
alphas_cold = [0.45, 0.00, 0.05, 0.50, 0.00]

# ==========================================
# 2. 构造冷/热专属验证矩阵
# ==========================================
def filter_urm_by_users(urm, mask):
    """保留 mask=True 的用户,其他用户的行清零"""
    new_urm = urm.copy().tocsr()
    users_to_zero = np.where(~mask)[0]

    for user_id in users_to_zero:
        start = new_urm.indptr[user_id]
        end = new_urm.indptr[user_id + 1]
        new_urm.data[start:end] = 0

    new_urm.eliminate_zeros()
    return new_urm

URM_test_cold = filter_urm_by_users(URM_validation_all, cold_mask)
URM_test_hot = filter_urm_by_users(URM_validation_all, hot_mask)

print(f"子评估器创建完成:")
print(f" - 冷用户验证集条目数: {URM_test_cold.nnz}")
print(f" - 热用户验证集条目数: {URM_test_hot.nnz}")

# ==========================================
# 3. 训练两个独立的模型
# ==========================================
print("\n" + "="*50)
print("训练冷用户模型...")
hybrid_cold = GeneralizedLinearHybrid(URM_train, recommender_list)
hybrid_cold.fit(alphas=alphas_cold)

print("训练热用户模型...")
hybrid_hot = GeneralizedLinearHybrid(URM_train, recommender_list)
hybrid_hot.fit(alphas=alphas_hot)

# ==========================================
# 4. 执行分群评估
# ==========================================
cutoff_list = [20]

print("正在执行分群 Recall@20 评估...")

# 冷用户评估
evaluator_cold = EvaluatorHoldout(URM_test_cold, cutoff_list=cutoff_list)
results_cold, _ = evaluator_cold.evaluateRecommender(hybrid_cold)  # ✅ 用 hybrid_cold
recall_cold = results_cold.loc[20]['RECALL']

# 热用户评估
evaluator_hot = EvaluatorHoldout(URM_test_hot, cutoff_list=cutoff_list)
results_hot, _ = evaluator_hot.evaluateRecommender(hybrid_hot)      # ✅ 用 hybrid_hot
recall_hot = results_hot.loc[20]['RECALL']

# ==========================================
# 5. 计算加权平均
# ==========================================
n_cold_val = (np.ediff1d(URM_test_cold.indptr) > 0).sum()
n_hot_val = (np.ediff1d(URM_test_hot.indptr) > 0).sum()

total_recall = (recall_cold * n_cold_val + recall_hot * n_hot_val) / (n_cold_val + n_hot_val)

print("-" * 40)
print(f"❄️  冷用户 Recall@20: {recall_cold:.6f} (n={n_cold_val})")
print(f"🔥 热用户 Recall@20: {recall_hot:.6f} (n={n_hot_val})")
print(f"🏆 组合策略总 Recall@20: {total_recall:.6f}")
print(f"📊 总验证用户数: {n_cold_val + n_hot_val}")
print("="*50)

子评估器创建完成:
 - 冷用户验证集条目数: 92533
 - 热用户验证集条目数: 516079

训练冷用户模型...
训练热用户模型...
正在执行分群 Recall@20 评估...
EvaluatorHoldout: Ignoring 14968 (55.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 12127 (100.0%) in 8.82 sec. Users per second: 1376
EvaluatorHoldout: Ignoring 12164 (44.9%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 14931 (100.0%) in 14.75 sec. Users per second: 1013
----------------------------------------
❄️  冷用户 Recall@20: 0.375975 (n=12127)
🔥 热用户 Recall@20: 0.225314 (n=14931)
🏆 组合策略总 Recall@20: 0.292838
📊 总验证用户数: 27058


In [88]:
# 冷/热用户在验证集的平均交互数
cold_avg_interactions = URM_test_cold.nnz / n_cold_val
hot_avg_interactions = URM_test_hot.nnz / n_hot_val

print(f"冷用户平均验证集交互: {cold_avg_interactions:.2f}")
print(f"热用户平均验证集交互: {hot_avg_interactions:.2f}")

冷用户平均验证集交互: 11.52
热用户平均验证集交互: 50.20


In [89]:
# 获取各群体的物品集合
cold_items = set(URM_test_cold.indices)
hot_items = set(URM_test_hot.indices)
popular_items = set(np.argsort(np.ediff1d(URM_train.tocsc().indptr))[-100:])  # Top 100

cold_popular_overlap = len(cold_items & popular_items) / len(cold_items)
hot_popular_overlap = len(hot_items & popular_items) / len(hot_items)

print(f"冷用户物品 & Top100 重叠率: {cold_popular_overlap:.2%}")
print(f"热用户物品 & Top100 重叠率: {hot_popular_overlap:.2%}")

冷用户物品 & Top100 重叠率: 1.49%
热用户物品 & Top100 重叠率: 1.45%


#### 6d.3 加载全量模型

In [22]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from Recommenders.BaseRecommender import BaseRecommender

# ==========================================
# 1. 定义支持归一化的混合类 (必须包含此定义)
# ==========================================
class GeneralizedLinearHybrid(BaseRecommender):
    """
    包含 Min-Max Normalization 的线性加权混合模型
    专门用于解决 EASE 与其他模型量级不一致的问题
    """
    def __init__(self, URM_train, recommenders: list, verbose=True):
        super(GeneralizedLinearHybrid, self).__init__(URM_train, verbose=verbose)
        self.recommenders = recommenders

    def fit(self, alphas: list = None):
        self.alphas = alphas

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        item_scores_total = None

        for index, recommender in enumerate(self.recommenders):
            # 获取原始分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # === Min-Max Normalization ===
            min_scores = scores.min(axis=1, keepdims=True)
            max_scores = scores.max(axis=1, keepdims=True)
            # 加上 1e-6 防止除以零
            norm_scores = (scores - min_scores) / (max_scores - min_scores + 1e-6)

            # 加权累加
            if item_scores_total is None:
                item_scores_total = norm_scores * self.alphas[index]
            else:
                item_scores_total += norm_scores * self.alphas[index]

        return item_scores_total

# ==========================================
# 2. 配置与文件路径
# ==========================================

# 最佳权重 (SLIM, vae, IALS)
# Best Alphas: SLIM=0.45, VAE=0.50, IALS=0.05
FINAL_ALPHAS = [0.40, 0.50, 0.04]

MODEL_FOLDER = 'model_output' # 模型存放路径
SUBMISSION_FOLDER = 'submission_output' # 结果存放路径

if not os.path.exists(SUBMISSION_FOLDER):
    os.makedirs(SUBMISSION_FOLDER)

# ==========================================
# 3. 加载全量数据训练好的模型
# ==========================================
print("\n--- 正在加载在全量数据 (URM_ALL) 上训练的模型... ---")

# 3.1 加载 SLIM
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all,
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 3.2 加载 vae (请确认文件名!)
final_recommender_vae = load_best_model(
    recommender_class=MultVAERecommender_PyTorch_OptimizerMask,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="29-MultVAE_Best_Model_Sub",
    modelfile_path=MODEL_FOLDER
)

# 3.3 加载 IALS
final_recommender_ials = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all,
    file_name="22-final_model_IALSRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# ==========================================
# 4. 构建混合模型并生成结果
# ==========================================

if final_recommender_slim and final_recommender_vae and final_recommender_ials:

    print(f"\n--- 构建 GeneralizedLinearHybrid 模型 ---")
    print(f"Weights -> SLIM: {FINAL_ALPHAS[0]}, EASE: {FINAL_ALPHAS[1]}, IALS: {FINAL_ALPHAS[2]}")

    # 按照权重对应的顺序放入模型列表
    final_recommender_list = [final_recommender_slim, final_recommender_vae, final_recommender_ials]

    # 初始化混合推荐器
    final_hybrid_recommender = GeneralizedLinearHybrid(urm_all, final_recommender_list)
    final_hybrid_recommender.fit(alphas=FINAL_ALPHAS)

    # --- 开始生成推荐 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终融合推荐... ---")

    for user_id in tqdm(target_user_ids):
        # 推荐并移除已见过的物品
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=20, # 比赛通常是 Top 20
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    filename_str = f"submission_FINAL_NormHybrid_S{FINAL_ALPHAS[0]}_E{FINAL_ALPHAS[1]}_I{FINAL_ALPHAS[2]}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, filename_str)

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! 🚀 ---")
    print(f"文件保存在: {SUBMISSION_PATH}")

else:
    print("\n>>> 错误：模型加载失败，请检查文件名和路径。")


--- 正在加载在全量数据 (URM_ALL) 上训练的模型... ---
--- 正在加载预训练模型: 5-1final_model_SLIMElasticNetRecommender-better ---
SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: 29-MultVAE_Best_Model_Sub ---
模型加载成功！

--- 正在加载预训练模型: 22-final_model_IALSRecommender-better ---
IALSRecommender: Loading model from file 'model_output22-final_model_IALSRecommender-better'


  0%|          | 36/27095 [00:00<01:16, 355.83it/s]

IALSRecommender: Loading complete
模型加载成功！


--- 构建 GeneralizedLinearHybrid 模型 ---
Weights -> SLIM: 0.4, EASE: 0.5, IALS: 0.04
--- 正在为 27095 名目标用户生成最终融合推荐... ---


100%|██████████| 27095/27095 [01:14<00:00, 362.11it/s]


--- 最终提交文件已成功生成! 🚀 ---
文件保存在: submission_output\submission_FINAL_NormHybrid_S0.4_E0.5_I0.04.csv


In [ ]:
# ==========================================
# Final Submission Script (5-Model Grand Hybrid)
# ==========================================


# 3.1 加载 SLIM
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all,
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 3.2 加载 EASE (请确认文件名!)
final_recommender_ease = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=urm_all,
    file_name="7-final_model_EASE_R_Recommender", # <--- 请确认这里是全量训练的模型文件名
    modelfile_path=MODEL_FOLDER
)

# 3.3 加载 IALS
final_recommender_ials = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all,
    file_name="22-final_model_IALSRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 3.4 加载 USER
final_recommender_user = load_best_model(
    recommender_class=UserKNNCFRecommender,
    urm_train=urm_all,
    file_name="17-final_model_UserKNNCFRecommender",
    modelfile_path=MODEL_FOLDER
)

# 3.5 加载 RP3
final_recommender_rp3 = load_best_model(
    recommender_class=RP3betaRecommender,
    urm_train=urm_all,
    file_name="3-1final_model_RP3betaRecommender",
    modelfile_path=MODEL_FOLDER
)

# 1. 填入你搜索到的最佳权重
# 示例：假设搜索结果是 [0.45, 0.25, 0.15, 0.10, 0.05]
FINAL_ALPHAS = [0.525, 0.225, 0.15, 0.05, 0.05] # <--- Replace this!

# 2. 准备全量模型列表 (全部基于 urm_all)
final_recommender_list = [
    final_recommender_slim, # 0
    final_recommender_ease, # 1
    final_recommender_ials, # 2
    final_recommender_user, # 3 (UserKNN)
    final_recommender_rp3   # 4 (RP3beta)
]

# 3. 初始化混合模型
final_hybrid = GeneralizedLinearHybrid(urm_all, final_recommender_list)
final_hybrid.fit(alphas=FINAL_ALPHAS)

# 4. 生成预测 (User Loop)
submission = []
print("Generating Grand Ensemble Recommendations...")
for user_id in tqdm(target_user_ids):
    recommended_items = final_hybrid.recommend(
        user_id,
        cutoff=20,
        remove_seen_flag=True
    )
    submission.append((user_id, ' '.join(map(str, recommended_items))))

# --- 保存提交文件 ---
filename_str = f"submission_5_NormHybrid.csv"
SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, filename_str)

df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 最终提交文件已成功生成! 🚀 ---")
print(f"文件保存在: {SUBMISSION_PATH}")

## 7. 生成最终提交文件
当你通过本地验证找到了最佳模型和参数后，在这里使用 **全部训练数据** (`urm_all`) 进行训练，并为测试集用户生成推荐。

### 7a. 重新训练以生成提交文件

In [ ]:
# --- 模型配置 ---
# 取消注释你想要使用的模型，并确保参数是最佳的

# # 配置
model_config = {
    "class": SLIMElasticNetRecommender,
    "params": {"topK": 421, "l1_ratio": 0.004959064325983541, "alpha": 0.0036221001361258906}
}


# 自动生成与模型相关的文件名，便于维护
model_name = model_config['class'].RECOMMENDER_NAME
submission_filename = f"submission_{model_name}.csv"
model_filename = f"final_model_{model_name}"
params_filename = f"final_params_{model_name}.json"

SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename)
MODEL_SAVE_PATH = os.path.join(MODEL_FOLDER, model_filename)
PARAMS_SAVE_PATH = os.path.join(MODEL_FOLDER, params_filename)


# -----------------------------------------------------------------------------
# STEP 2: 执行生成流程 (通常无需修改以下代码)
# -----------------------------------------------------------------------------

# 确保输出和模型文件夹存在
for folder in [MODEL_FOLDER, SUBMISSION_FOLDER]:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"文件夹 '{folder}' 已创建。")

# 1. 确定最终模型类和参数
final_model_class = model_config["class"]
final_model_params = model_config["params"]

# 2. 在 *全部* 数据上训练最终模型
print(f"--- 正在使用模型 '{model_name}' 在全量数据上进行训练... ---")
# 警告：对于 SLIMElasticNet 等模型，这一步会非常耗时!
final_recommender = final_model_class(urm_all)
final_recommender.fit(**final_model_params)
print("最终模型训练完成。\n")


# 3. ***** 新增功能: 保存模型和参数 *****
print(f"--- 正在保存已训练的模型和参数... ---")
# 保存模型对象
final_recommender.save_model(folder_path=MODEL_FOLDER, file_name=model_filename)
# 保存参数字典为 JSON 文件
with open(PARAMS_SAVE_PATH, 'w') as f:
    json.dump(final_model_params, f, indent=4)
print(f"模型已保存至: {MODEL_SAVE_PATH}.zip")
print(f"参数已保存至: {PARAMS_SAVE_PATH}\n")


# 4. 加载需要预测的目标用户
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

# 5. 生成推荐并保存到文件
print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
submission = []
for user_id in tqdm(target_user_ids):
    recommended_items = final_recommender.recommend(
        user_id,
        cutoff=EVALUATION_CUTOFF,
        remove_seen_flag=True
    )
    submission.append((user_id, ' '.join(map(str, recommended_items))))

# 6. 创建DataFrame并保存为CSV
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 提交文件已成功生成! ---")
print(f"文件保存在: {SUBMISSION_PATH}")
print("\n文件预览:")
print(df_submission.head())

### 7b. 读取模型以获得提交文件

In [ ]:
# --- STEP 1: 加载预训练的最佳模型 ---

# 使用我们之前编写的 load_best_model 辅助函数
# 注意：这里传入的是 urm_all，因为 recommend 函数需要访问完整的交互历史来移除已看过的物品
# 而模型本身是在 URM_train 上训练的，这些信息都保存在了模型文件中。
# 实际上，初始化时传入哪个 URM 并不影响加载，但传入 urm_all 在逻辑上更清晰。
final_recommender = load_best_model(SLIMElasticNetRecommender, urm_all)


if final_recommender:
    # --- STEP 2: 执行生成流程 ---

    # 1. 加载需要预测的目标用户
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    # 2. 生成推荐并保存到文件
    submission = []
    # 为了让输出更美观，我们可以使用 tqdm 来显示进度条
    from tqdm import tqdm

    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # 3. 创建DataFrame并保存为CSV
    submission_filename = f"submission_{final_recommender.RECOMMENDER_NAME}_optimized.csv"
    SUBMISSION_PATH = os.path.join(OUTPUT_FOLDER, submission_filename)

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
    print("\n文件预览:")
    print(df_submission.head())

else:
    print(">>> 模型加载失败，无法生成提交文件。请检查文件路径和名称。")